#### Surface EOF's

In [15]:
#!/usr/bin/env python3
# ============================================================
# Combined SURFACE EOF Analysis for CMIP6 ocean variables in (lat,lon)
# over the masked Atlantic domain
#
# EOFs fit on TRAIN only; PCs are projected for FULL period.
# ============================================================

import os
import glob
import re
import numpy as np
import xarray as xr
from eofs.standard import Eof
import warnings
warnings.filterwarnings("ignore")

# Prevent HDF5 file locking issues on shared filesystems
os.environ["HDF5_USE_FILE_LOCKING"] = "FALSE"

# ============================================================
# CONFIG
# ============================================================
MODEL   = "MPI-ESM1-2-LR"   # EC-Earth3, IPSL-CM6A-LR, CESM2, MPI-ESM1-2-LR
VAR     = "so"            # thetao or so
NMODES  = 10

START_YEAR = 1850
END_YEAR   = 2014

# TRAIN split on the COMMON time axis
TRAIN_FRACTION = 0.85

# If True: subtract temporal mean at each grid point over FULL period
# before TRAIN-only EOF fitting (same philosophy as your current script)
REMOVE_TEMPORAL_MEAN = True

# ============================================================
# DIRECTORIES
# ============================================================
if MODEL == "EC-Earth3":
    IN_DIR = f"/data/projects/nckf/frekle/CMIP6_data/EC-Earth3/{VAR}/"
elif MODEL == "IPSL-CM6A-LR":
    IN_DIR = f"/data/projects/nckf/frekle/CMIP6_data/IPSL-CM6A-LR/{VAR}/masked/"
elif MODEL == "CESM2":
    IN_DIR = f"/data/projects/nckf/frekle/CMIP6_data/CESM2/{VAR}/masked/"
elif MODEL == "MPI-ESM1-2-LR":
    IN_DIR = f"/data/projects/nckf/frekle/CMIP6_data/MPI-ESM1-2-LR/{VAR}/masked/"
else:
    raise ValueError("Unknown MODEL")

OUT_DIR = f"/data/projects/nckf/frekle/EOF_results/{MODEL}/surface_latlon/Train_period_{int(TRAIN_FRACTION*100)}pct/"
os.makedirs(OUT_DIR, exist_ok=True)

# ============================================================
# UTILITIES
# ============================================================

def parse_member_label(path):
    m = re.search(r"_(r\d+i\d+p\d+f\d+)", os.path.basename(path))
    return m.group(1) if m else os.path.basename(path)

def wrap_lon(lon):
    return ((lon + 180) % 360) - 180

def standardize_latlon(ds):
    """
    Convert model-specific horizontal coordinates to:
      - dims: y, x
      - coords/variables: lat2d(y,x), lon2d(y,x)

    Works for both rectilinear and curvilinear grids.
    """
    lat_var = next(v for v in ["lat", "latitude", "nav_lat"] if v in ds)
    lon_var = next(v for v in ["lon", "longitude", "nav_lon"] if v in ds)

    lat = ds[lat_var]
    lon = ds[lon_var]

    # --------------------------------------------------------
    # Case 1: rectilinear grid -> lat(y), lon(x)
    # --------------------------------------------------------
    if lat.ndim == 1 and lon.ndim == 1:
        ydim = lat.dims[0]
        xdim = lon.dims[0]

        # rename dims only if needed
        dim_map = {}
        if ydim != "y":
            dim_map[ydim] = "y"
        if xdim != "x":
            dim_map[xdim] = "x"
        if dim_map:
            ds = ds.rename_dims(dim_map)

        # refresh objects after possible dim rename
        lat = ds[lat_var]
        lon = ds[lon_var]

        lon2d, lat2d = np.meshgrid(lon.values, lat.values)

        # rename original coordinate variables only if needed
        var_map = {}
        if lat_var != "lat_old":
            var_map[lat_var] = "lat_old"
        if lon_var != "lon_old":
            var_map[lon_var] = "lon_old"
        ds = ds.rename(var_map)

        ds["lat2d"] = xr.DataArray(lat2d, dims=("y", "x"))
        ds["lon2d"] = xr.DataArray(lon2d, dims=("y", "x"))
        return ds

    # --------------------------------------------------------
    # Case 2: curvilinear grid -> lat(y,x), lon(y,x)
    # --------------------------------------------------------
    elif lat.ndim == 2 and lon.ndim == 2:
        ydim, xdim = lat.dims

        # rename dims only if needed
        dim_map = {}
        if ydim != "y":
            dim_map[ydim] = "y"
        if xdim != "x":
            dim_map[xdim] = "x"
        if dim_map:
            ds = ds.rename_dims(dim_map)

        # rename lat/lon variables only if needed
        var_map = {}
        if lat_var != "lat2d":
            var_map[lat_var] = "lat2d"
        if lon_var != "lon2d":
            var_map[lon_var] = "lon2d"
        if var_map:
            ds = ds.rename(var_map)

        return ds

    else:
        raise ValueError(
            f"Unsupported lat/lon coordinate structure: "
            f"lat.ndim={lat.ndim}, lon.ndim={lon.ndim}, "
            f"lat.dims={lat.dims}, lon.dims={lon.dims}"
        )

def get_depth_dim_and_coord(da, ds):
    """
    Return vertical dimension name and depth values in meters.
    If no vertical dimension exists, return (None, None).
    """
    if "olevel" in da.dims:
        return "olevel", np.asarray(ds["olevel"].values, dtype=float)
    if "depth" in da.dims:
        return "depth", np.asarray(ds["depth"].values, dtype=float)
    if "lev" in da.dims:
        z = np.asarray(ds["lev"].values, dtype=float)
        # CESM often stores lev in centimeters
        if np.nanmax(z) > 10000:
            z = z / 100.0
        return "lev", z
    return None, None

def select_surface_field(da, ds):
    """
    Select the shallowest model level if a vertical dimension exists.
    Returns:
      da_surf  : (time, y, x)
      surf_z_m : scalar depth in meters, or NaN if no z dimension exists
      z_name   : vertical dim name or None
    """
    z_name, z = get_depth_dim_and_coord(da, ds)

    if z_name is None:
        # already 2D horizontal field
        return da, np.nan, None

    iz = int(np.nanargmin(z))
    surf_z_m = float(z[iz])

    da_surf = da.isel({z_name: iz})
    return da_surf, surf_z_m, z_name

def make_space_mask_from_train(data_train):
    """
    data_train shape: (member, time, y, x)
    Returns a boolean mask of spatial points valid for ALL train samples.
    This guarantees a constant mask for EOF solving.
    """
    valid_space = np.all(np.isfinite(data_train), axis=(0, 1))
    return valid_space

def flatten_valid_space(data4d, valid_space):
    """
    data4d shape: (member, time, y, x)
    valid_space shape: (y, x)
    Returns 2D array: (member*time, n_valid_space)
    """
    n_member, n_time, ny, nx = data4d.shape
    flat = data4d.reshape(n_member * n_time, ny * nx)
    keep = valid_space.reshape(ny * nx)
    return flat[:, keep]

def unflatten_eofs_to_fullgrid(eofs_valid, valid_space, ny, nx):
    """
    eofs_valid shape: (mode, n_valid_space)
    Returns full-grid EOFs: (mode, ny, nx) with NaN over invalid cells.
    """
    nmode = eofs_valid.shape[0]
    out = np.full((nmode, ny * nx), np.nan, dtype=float)
    out[:, valid_space.reshape(ny * nx)] = eofs_valid
    return out.reshape(nmode, ny, nx)

# ============================================================
# MAIN WORKFLOW
# ============================================================

def run_surface_eof():

    print("\n" + "="*80)
    print(f"▶ SURFACE LAT-LON EOF | MODEL={MODEL} | VAR={VAR}")
    print("="*80)

    files = sorted(glob.glob(os.path.join(IN_DIR, f"*{VAR}*_masked*.nc")))
    if not files:
        raise FileNotFoundError(f"No masked files found in {IN_DIR}")

    fields = []
    member_labels = []

    lat_ref = None
    lon_ref = None
    surf_depths = []

    for i, f in enumerate(files, start=1):
        label = parse_member_label(f)
        member_labels.append(label)

        print(f"\n  → Member {i}/{len(files)}: {label}")

        ds = standardize_latlon(xr.open_dataset(f))
        da = ds[VAR]

        da = da.sel(time=slice(f"{START_YEAR}-01-01", f"{END_YEAR}-12-31"))

        da_surf, surf_z_m, z_name = select_surface_field(da, ds)
        surf_depths.append(surf_z_m)

        # Ensure standard order
        da_surf = da_surf.transpose("time", "y", "x")

        if REMOVE_TEMPORAL_MEAN:
            da_surf = da_surf - da_surf.mean("time")

        da_surf = da_surf.expand_dims(member=[label])
        fields.append(da_surf)

        if lat_ref is None:
            lat_ref = ds["lat2d"].values
            lon_ref = wrap_lon(ds["lon2d"].values)

        print(f"    Surface depth used: {surf_z_m:.6f} m" if np.isfinite(surf_z_m) else "    No vertical dim found")

        ds.close()

    print("\n▶ Aligning time across members")
    common_time = fields[0]["time"].values
    for s in fields[1:]:
        common_time = np.intersect1d(common_time, s["time"].values)

    common_time = np.sort(common_time)

    combined = xr.concat([s.sel(time=common_time) for s in fields], dim="member")

    n_member = combined.sizes["member"]
    n_time   = combined.sizes["time"]
    ny       = combined.sizes["y"]
    nx       = combined.sizes["x"]

    # TRAIN split on common axis
    n_train = int(np.floor(TRAIN_FRACTION * n_time))
    n_train = max(2, min(n_train, n_time))
    train_time = combined["time"].values[:n_train]

    train_mask = np.zeros(n_time, dtype=bool)
    train_mask[:n_train] = True

    print(f"\n▶ Combined shape: member={n_member}, time={n_time}, y={ny}, x={nx}")
    print(f"▶ TRAIN_FRACTION={TRAIN_FRACTION} -> n_train={n_train}/{n_time} "
          f"(train end = {str(train_time[-1])})")

    # Full and train data
    data_full  = combined.transpose("member", "time", "y", "x").values.astype(float)
    data_train = combined.sel(time=train_time).transpose("member", "time", "y", "x").values.astype(float)

    print("▶ Building constant spatial mask from TRAIN data")
    valid_space = make_space_mask_from_train(data_train)
    n_valid = int(valid_space.sum())

    if n_valid == 0:
        raise ValueError("No spatial points remain after TRAIN valid-space masking")

    print(f"▶ Valid ocean points kept for EOF: {n_valid} / {ny*nx}")

    data2d_train = flatten_valid_space(data_train, valid_space)
    data2d_full  = flatten_valid_space(data_full,  valid_space)

    data2d_train = np.ma.masked_invalid(data2d_train)
    data2d_full  = np.ma.masked_invalid(data2d_full)

    print("▶ Computing EOF weights")
    lat_valid = lat_ref[valid_space]
    weights = np.sqrt(np.clip(np.cos(np.deg2rad(lat_valid)), 0, None))

    print("▶ Solving EOFs on TRAIN only")
    solver = Eof(data2d_train, weights=weights)

    eofs_valid = np.asarray(solver.eofs(neofs=NMODES))
    vf = np.asarray(solver.varianceFraction()[:NMODES])

    print("▶ Projecting FULL period onto TRAIN EOFs")
    pcs_full_flat = np.asarray(solver.projectField(data2d_full, neofs=NMODES))
    pcs = pcs_full_flat.reshape(n_member, n_time, NMODES)

    print("▶ Restoring EOF maps to full grid")
    EOFs = unflatten_eofs_to_fullgrid(eofs_valid, valid_space, ny, nx)

    surface_depth_mean = float(np.nanmean(surf_depths))
    surface_depth_min  = float(np.nanmin(surf_depths))
    surface_depth_max  = float(np.nanmax(surf_depths))

    outfile = os.path.join(OUT_DIR, f"EOF_surface_latlon_{VAR}.nc")
    print(f"▶ Writing {outfile}")

    xr.Dataset(
        {
            "EOF": (("mode", "y", "x"), EOFs),
            "PC":  (("member", "time", "mode"), pcs),
            "variance_fraction": (("mode",), vf),
            "train_mask": (("time",), train_mask),
            "valid_mask": (("y", "x"), valid_space.astype(np.int8)),
        },
        coords={
            "mode": np.arange(1, NMODES + 1),
            "member": np.array(member_labels, dtype=object),
            "time": combined["time"].values,
            "lat": (("y", "x"), lat_ref),
            "lon": (("y", "x"), lon_ref),
        },
        attrs={
            "MODEL": MODEL,
            "VAR": VAR,
            "GRID": "surface_latlon",
            "WEIGHTING": "sqrt(cos(lat))",
            "EOF_FIT": "TRAIN_ONLY",
            "TRAIN_FRACTION": float(TRAIN_FRACTION),
            "TRAIN_START": str(train_time[0]),
            "TRAIN_END": str(train_time[-1]),
            "START_YEAR": int(START_YEAR),
            "END_YEAR": int(END_YEAR),
            "REMOVE_TEMPORAL_MEAN": str(bool(REMOVE_TEMPORAL_MEAN)),
            "SURFACE_LEVEL_SELECTION": "shallowest_available_model_level",
            "SURFACE_DEPTH_MEAN_M": surface_depth_mean,
            "SURFACE_DEPTH_MIN_M": surface_depth_min,
            "SURFACE_DEPTH_MAX_M": surface_depth_max,
        }
    ).to_netcdf(outfile)

    print("✓ Done")

# ============================================================
# RUN
# ============================================================

if __name__ == "__main__":
    print(f"\n=== MODEL={MODEL} | VAR={VAR} ===")
    print(f"TRAIN_FRACTION={TRAIN_FRACTION}")
    print(f"REMOVE_TEMPORAL_MEAN={REMOVE_TEMPORAL_MEAN}")

    run_surface_eof()

    print("\n✅ Surface lat-lon EOF processed successfully.")


=== MODEL=MPI-ESM1-2-LR | VAR=so ===
TRAIN_FRACTION=0.85
REMOVE_TEMPORAL_MEAN=True

▶ SURFACE LAT-LON EOF | MODEL=MPI-ESM1-2-LR | VAR=so

  → Member 1/30: r10i1p1f1
    Surface depth used: 6.000000 m

  → Member 2/30: r11i1p1f1
    Surface depth used: 6.000000 m

  → Member 3/30: r12i1p1f1
    Surface depth used: 6.000000 m

  → Member 4/30: r13i1p1f1
    Surface depth used: 6.000000 m

  → Member 5/30: r14i1p1f1
    Surface depth used: 6.000000 m

  → Member 6/30: r15i1p1f1
    Surface depth used: 6.000000 m

  → Member 7/30: r16i1p1f1
    Surface depth used: 6.000000 m

  → Member 8/30: r17i1p1f1
    Surface depth used: 6.000000 m

  → Member 9/30: r18i1p1f1
    Surface depth used: 6.000000 m

  → Member 10/30: r19i1p1f1
    Surface depth used: 6.000000 m

  → Member 11/30: r1i1p1f1
    Surface depth used: 6.000000 m

  → Member 12/30: r20i1p1f1
    Surface depth used: 6.000000 m

  → Member 13/30: r21i1p1f1
    Surface depth used: 6.000000 m

  → Member 14/30: r22i1p1f1
    Surface

### Making the ranking file

In [ ]:
#!/usr/bin/env python3
"""
MASTER WORKFLOW (SURFACE LAT-LON EOF VERSION, MODEL-GENERIC, ENSMEAN OR MEMBER)
================================================================================
Produces, in ONE consistent run per model or member:

(1) feature_ranking_pre.csv
    - correlation ranking computed on PRE period only (no post leakage)

(2) spike_curve_post.csv + spike_plot + spikes_top10.txt + spike_features.csv
    - spike curve computed on POST period (target regime)
    - alpha chosen by TimeSeriesSplit CV *within POST* for a chosen n_for_alpha
    - spikes extracted based on ΔR² threshold

Supports:
- MODE = "ensmean"  -> use ensemble-mean PCs and ensemble-mean AMOC target
- MODE = "member"   -> use member-specific PCs and member-specific AMOC target

You can either:
- choose one specific member with member_id = "r1i1p1f1"
- or run all members automatically with RUN_ALL_MEMBERS = True

Requirements:
- EOF_surface_latlon_{var}.nc files with PC variable covering full period
- AMOC_{MODEL}.nc with variable TARGET
"""

import os
import numpy as np
import pandas as pd
import xarray as xr
import matplotlib.pyplot as plt

from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import Ridge
from sklearn.metrics import r2_score
from sklearn.model_selection import TimeSeriesSplit

os.environ["HDF5_USE_FILE_LOCKING"] = "FALSE"

# ============================================================
# USER SETTINGS
# ============================================================
MODEL   = "CESM2"   # "EC-Earth3", "IPSL-CM6A-LR", "CESM2", "MPI-ESM1-2-LR"

# AMOC target variable in AMOC file
# examples:
#   "AMOC_45N_ensmean"
#   "AMOC_45N_member"
TARGET  = "AMOC_45N_ensmean"

# MODE:
#   "ensmean" -> ensemble mean
#   "member"  -> one member or all members
MODE = "ensmean"

# If MODE="member":
#   choose one specific member, e.g. "r1i1p1f1"
#   or leave as None and set RUN_ALL_MEMBERS=True
member_id = None

# If MODE="member" and member_id is None:
#   True  -> run all members automatically
#   False -> error
RUN_ALL_MEMBERS = True

EOF_DIR   = f"/data/projects/nckf/frekle/EOF_results/{MODEL}/surface_latlon/Train_period_85pct/"
AMOC_FILE = f"/data/users/frekle/AMOC_analysis/AMOC_{MODEL}.nc"

BASE_OUTDIR = f"/data/users/frekle/Final_figures/{MODEL}/{TARGET}/Feature_selection_surface/"
os.makedirs(BASE_OUTDIR, exist_ok=True)

VARS    = ["thetao", "so"]
N_MODES = 10

YEAR_START = 1850
YEAR_END   = 2014

MAX_LAG_ALLOWED = 20
STANDARDIZE_PC = True

R2_EVOL_MAX_RANKED = 60
SPIKE_THRESH = 0.01

# Alpha tuning on POST
ALPHA_GRID = [0.01, 0.03, 0.1, 0.3, 1, 3, 4, 5, 10, 30, 40, 50, 60, 70, 100, 300]
ALPHA_CV_N_SPLITS = 4
ALPHA_CV_MIN_TRAIN = 8
ALPHA_CV_MIN_VAL   = 4

# Choose alpha using this many ranked features
N_FOR_ALPHA = 5

# ============================================================
# Helper functions
# ============================================================

def extract_years(time_coord):
    try:
        return xr.DataArray(time_coord).dt.year.values.astype(int)
    except Exception:
        return np.array([int(str(x)[:4]) for x in np.asarray(time_coord)], dtype=int)

def corr_1d(a, b):
    m = np.isfinite(a) & np.isfinite(b)
    if m.sum() < 3:
        return np.nan
    aa = a[m] - np.mean(a[m])
    bb = b[m] - np.mean(b[m])
    denom = np.sqrt(np.sum(aa**2) * np.sum(bb**2))
    if denom == 0:
        return np.nan
    return np.sum(aa * bb) / denom

def get_example_eof_file():
    f = os.path.join(EOF_DIR, f"EOF_surface_latlon_{VARS[0]}.nc")
    if not os.path.exists(f):
        raise FileNotFoundError(f"Could not find EOF file: {f}")
    return f

def get_train_end():
    f = get_example_eof_file()
    ds = xr.open_dataset(f)
    y = int(str(ds.attrs["TRAIN_END"])[:4])
    ds.close()
    return y

def get_member_list():
    f = get_example_eof_file()
    ds = xr.open_dataset(f)
    if "member" not in ds["PC"].dims:
        ds.close()
        raise ValueError("EOF file has no member dimension, so MODE='member' is not possible.")
    members = [str(m) for m in ds["member"].values]
    ds.close()
    return members

def make_outdir(mode, member_id=None):
    if mode == "ensmean":
        outdir = os.path.join(BASE_OUTDIR, "ensmean")
    elif mode == "member":
        if member_id is None:
            raise ValueError("member_id must be provided when mode='member'")
        outdir = os.path.join(BASE_OUTDIR, member_id)
    else:
        raise ValueError("mode must be 'ensmean' or 'member'")
    os.makedirs(outdir, exist_ok=True)
    return outdir

def load_pc(var, mode="ensmean", member_id=None):
    f = os.path.join(EOF_DIR, f"EOF_surface_latlon_{var}.nc")
    if not os.path.exists(f):
        raise FileNotFoundError(f"Missing EOF file: {f}")

    ds = xr.open_dataset(f)

    pc_name = "PC" if "PC" in ds.data_vars else "pcs" if "pcs" in ds.data_vars else None
    if pc_name is None:
        ds.close()
        raise KeyError(f"No PC variable found in {f}. Expected 'PC' or 'pcs'.")

    PC = ds[pc_name].isel(mode=slice(0, N_MODES))

    if "member" in PC.dims:
        if mode == "ensmean":
            PC = PC.mean("member")
        elif mode == "member":
            if member_id is None:
                raise ValueError("member_id must be provided when mode='member'")
            member_vals = [str(m) for m in PC["member"].values]
            if member_id not in member_vals:
                raise ValueError(f"member_id '{member_id}' not found in {f}")
            PC = PC.sel(member=member_id)
        else:
            raise ValueError("mode must be 'ensmean' or 'member'")

    PC = PC.transpose("time", "mode")
    years = extract_years(PC["time"])
    PC = PC.assign_coords(year=("time", years)).swap_dims({"time": "year"}).drop_vars("time")

    ds.close()
    return PC.astype(float)

def load_amoc(mode="ensmean", member_id=None):
    if not os.path.exists(AMOC_FILE):
        raise FileNotFoundError(f"Missing AMOC file: {AMOC_FILE}")

    ds = xr.open_dataset(AMOC_FILE)
    if TARGET not in ds:
        raise KeyError(f"TARGET '{TARGET}' not found in AMOC file")

    y = ds[TARGET].squeeze()

    if "member" in y.dims:
        if mode == "ensmean":
            y = y.mean("member")

        elif mode == "member":
            if member_id is None:
                raise ValueError("member_id must be provided when mode='member'")

            if "member" in y.coords:
                member_vals = [str(m) for m in y["member"].values]
                if member_id in member_vals:
                    y = y.sel(member=member_id)
                else:
                    eof_members = get_member_list()
                    if member_id not in eof_members:
                        raise ValueError(f"member_id '{member_id}' not found in EOF member list")
                    idx = eof_members.index(member_id)
                    if idx >= y.sizes["member"]:
                        raise ValueError(f"member index {idx} out of bounds for AMOC file")
                    y = y.isel(member=idx)
            else:
                eof_members = get_member_list()
                if member_id not in eof_members:
                    raise ValueError(f"member_id '{member_id}' not found in EOF member list")
                idx = eof_members.index(member_id)
                if idx >= y.sizes["member"]:
                    raise ValueError(f"member index {idx} out of bounds for AMOC file")
                y = y.isel(member=idx)

        else:
            raise ValueError("mode must be 'ensmean' or 'member'")

    y = y.squeeze(drop=True)

    if "year" not in y.dims:
        years = extract_years(y["time"])
        y = y.assign_coords(year=("time", years)).swap_dims({"time": "year"}).drop_vars("time")

    ds.close()
    return y.astype(float)

def save_run_info(outdir, mode, member_id, train_end_year, years, idx_pre, idx_post):
    info = {
        "MODEL": MODEL,
        "TARGET": TARGET,
        "MODE": mode,
        "member_id": str(member_id) if member_id is not None else "None",
        "TRAIN_END_YEAR": int(train_end_year),
        "YEAR_START_USED": int(years[0]),
        "YEAR_END_USED": int(years[-1]),
        "PRE_START": int(years[idx_pre[0]]) if len(idx_pre) > 0 else np.nan,
        "PRE_END": int(years[idx_pre[-1]]) if len(idx_pre) > 0 else np.nan,
        "POST_START": int(years[idx_post[0]]) if len(idx_post) > 0 else np.nan,
        "POST_END": int(years[idx_post[-1]]) if len(idx_post) > 0 else np.nan,
        "N_YEARS_TOTAL": int(len(years)),
        "N_YEARS_PRE": int(len(idx_pre)),
        "N_YEARS_POST": int(len(idx_post)),
        "N_MODES": int(N_MODES),
        "MAX_LAG_ALLOWED": int(MAX_LAG_ALLOWED),
        "STANDARDIZE_PC": bool(STANDARDIZE_PC),
        "R2_EVOL_MAX_RANKED": int(R2_EVOL_MAX_RANKED),
        "SPIKE_THRESH": float(SPIKE_THRESH),
        "N_FOR_ALPHA": int(N_FOR_ALPHA),
        "EOF_KIND": "surface_latlon",
    }
    pd.DataFrame([info]).to_csv(os.path.join(outdir, "run_info.csv"), index=False)

# ============================================================
# Core workflow
# ============================================================

def run_feature_workflow(mode="ensmean", member_id=None):
    outdir = make_outdir(mode, member_id)

    print("\n" + "=" * 90)
    print(f"RUNNING FEATURE WORKFLOW | MODEL={MODEL} | TARGET={TARGET} | MODE={mode} | member={member_id}")
    print("=" * 90)

    train_end_year = get_train_end()
    print("PRE ends at:", train_end_year)

    # ----------------------------
    # Load target + PCs
    # ----------------------------
    amoc = load_amoc(mode=mode, member_id=member_id)
    pc_dict = {v: load_pc(v, mode=mode, member_id=member_id) for v in VARS}

    # ----------------------------
    # Align common years
    # ----------------------------
    years = amoc["year"].values.astype(int)
    for da in pc_dict.values():
        years = np.intersect1d(years, da["year"].values.astype(int))

    years = years[(years >= YEAR_START) & (years <= YEAR_END)]
    years = np.sort(years)

    if len(years) == 0:
        raise ValueError("No overlapping years found between AMOC and PCs.")

    y = amoc.sel(year=years).values.astype(float)
    PC = {k: v.sel(year=years).values.astype(float) for k, v in pc_dict.items()}

    idx_pre  = np.where(years <= train_end_year)[0]
    idx_post = np.where(years >  train_end_year)[0]

    if len(idx_pre) == 0:
        raise ValueError("PRE period is empty.")
    if len(idx_post) == 0:
        raise ValueError("POST period is empty.")

    print("PRE :", years[idx_pre[0]],  "-", years[idx_pre[-1]],  f"({len(idx_pre)} years)")
    print("POST:", years[idx_post[0]], "-", years[idx_post[-1]], f"({len(idx_post)} years)")

    save_run_info(outdir, mode, member_id, train_end_year, years, idx_pre, idx_post)

    # ============================================================
    # 1) Ranking on PRE
    # ============================================================
    print("▶ Building PRE correlation ranking")

    rows = []
    for var in VARS:
        X = PC[var][idx_pre, :]   # shape: (n_pre, N_MODES)

        if STANDARDIZE_PC:
            mu = np.nanmean(X, axis=0)
            sd = np.nanstd(X, axis=0)
            X = (X - mu) / (sd + 1e-12)

        ypre = y[idx_pre]

        for lag in range(MAX_LAG_ALLOWED + 1):
            if lag >= len(ypre):
                continue
            t = np.arange(lag, len(ypre))
            for m0 in range(N_MODES):
                c = corr_1d(X[t - lag, m0], ypre[t])
                if np.isfinite(c):
                    rows.append(
                        dict(
                            var=var,
                            mode=m0 + 1,
                            lag=lag,
                            corr=c,
                            abs_corr=abs(c),
                        )
                    )

    if len(rows) == 0:
        raise ValueError("No valid PRE correlations were computed.")

    df_rank = pd.DataFrame(rows).sort_values("abs_corr", ascending=False).reset_index(drop=True)
    df_rank.to_csv(os.path.join(outdir, "feature_ranking_pre.csv"), index=False)
    print(f"✅ Saved: {os.path.join(outdir, 'feature_ranking_pre.csv')}")

    # ranked feature tuples: (var, mode0, lag)
    feats_ranked = [
        (r.var, int(r.mode - 1), int(r.lag))
        for r in df_rank.itertuples(index=False)
    ]

    # ============================================================
    # 2) Alpha selection on POST
    # ============================================================
    print("▶ Selecting alpha on POST by TimeSeriesSplit CV")

    def build_X_post(n):
        feats = feats_ranked[:n]
        if len(feats) == 0:
            raise ValueError("No features available to build X_post.")
        maxlag = max(f[2] for f in feats)
        usable = idx_post[idx_post >= maxlag]

        Xmat = np.zeros((len(usable), len(feats)), dtype=float)
        for i, t in enumerate(usable):
            for j, (v, m, lag) in enumerate(feats):
                Xmat[i, j] = PC[v][t - lag, m]
        return Xmat, y[usable], usable

    n_for_alpha = min(N_FOR_ALPHA, len(feats_ranked))
    X_post, y_post, usable_post = build_X_post(n_for_alpha)

    if len(y_post) < (ALPHA_CV_MIN_TRAIN + ALPHA_CV_MIN_VAL + 1):
        raise ValueError(
            f"POST period too short for alpha CV. "
            f"Need at least ~{ALPHA_CV_MIN_TRAIN + ALPHA_CV_MIN_VAL + 1} samples, got {len(y_post)}."
        )

    tscv = TimeSeriesSplit(n_splits=ALPHA_CV_N_SPLITS)
    alpha_scores = []

    for a in ALPHA_GRID:
        scores = []
        for tr, va in tscv.split(X_post):
            if len(tr) < ALPHA_CV_MIN_TRAIN or len(va) < ALPHA_CV_MIN_VAL:
                continue

            mdl = make_pipeline(StandardScaler(), Ridge(alpha=a))
            mdl.fit(X_post[tr], y_post[tr])
            pred = mdl.predict(X_post[va])
            scores.append(r2_score(y_post[va], pred))

        if len(scores) > 0:
            alpha_scores.append(
                dict(
                    alpha=float(a),
                    mean_cv_r2=float(np.mean(scores)),
                    std_cv_r2=float(np.std(scores)),
                    n_folds_used=int(len(scores)),
                )
            )

    if len(alpha_scores) == 0:
        raise ValueError("No valid alpha CV scores were computed.")

    df_alpha = pd.DataFrame(alpha_scores).sort_values("mean_cv_r2", ascending=False).reset_index(drop=True)
    best_alpha = float(df_alpha.iloc[0]["alpha"])

    df_alpha.to_csv(os.path.join(outdir, "alpha_cv_post_grid.csv"), index=False)
    pd.DataFrame([dict(best_alpha_post=best_alpha)]).to_csv(
        os.path.join(outdir, "best_alpha_post.csv"),
        index=False
    )

    print("Best alpha (POST CV):", best_alpha)

    # ============================================================
    # 3) Spike curve on POST
    # ============================================================
    print("▶ Building POST spike curve")

    curve = []
    prev = None
    spikes = []

    nmax = min(R2_EVOL_MAX_RANKED, len(feats_ranked))

    for n in range(1, nmax + 1):
        Xp, yp, usable = build_X_post(n)
        mdl = make_pipeline(StandardScaler(), Ridge(alpha=best_alpha))
        mdl.fit(Xp, yp)
        pred = mdl.predict(Xp)

        r2 = r2_score(yp, pred)
        dr2 = np.nan if prev is None else (r2 - prev)

        curve.append(
            dict(
                n_features=n,
                post_r2=float(r2),
                delta_r2=np.nan if not np.isfinite(dr2) else float(dr2),
            )
        )

        if np.isfinite(dr2) and dr2 >= SPIKE_THRESH:
            spikes.append((n, float(dr2), float(r2)))

        prev = r2

    df_curve = pd.DataFrame(curve)
    df_curve.to_csv(os.path.join(outdir, "spike_curve_post.csv"), index=False)

    # spike feature rows
    spike_rows = []
    for (n, dr2, r2) in spikes:
        v, m, lag = feats_ranked[n - 1]
        spike_rows.append(
            dict(
                n=n,
                delta_r2=dr2,
                post_r2=r2,
                var=v,
                mode=m + 1,
                lag=lag,
            )
        )

    if len(spike_rows) > 0:
        df_spikes = pd.DataFrame(spike_rows).sort_values("delta_r2", ascending=False).reset_index(drop=True)
    else:
        df_spikes = pd.DataFrame(columns=["n", "delta_r2", "post_r2", "var", "mode", "lag"])

    df_spikes.to_csv(os.path.join(outdir, "spike_features_post.csv"), index=False)

    # top10 text file
    with open(os.path.join(outdir, "spikes_top10_post.txt"), "w") as f:
        if len(spikes) == 0:
            f.write("No spikes found above threshold.\n")
        else:
            for (n, dr2, r2) in sorted(spikes, key=lambda x: x[1], reverse=True)[:10]:
                v, m, lag = feats_ranked[n - 1]
                f.write(
                    f"n={n:3d}  ΔR²={dr2:+.3f}  R²={r2:.3f}  "
                    f"| feature: var={v}, mode={m+1}, lag={lag}\n"
                )

    print("\nTop spikes (POST):")
    if len(spikes) == 0:
        print("No spikes found above threshold.")
    else:
        for (n, dr2, r2) in sorted(spikes, key=lambda x: x[1], reverse=True)[:10]:
            v, m, lag = feats_ranked[n - 1]
            print(
                f"n={n:3d}  ΔR²={dr2:+.3f}  R²={r2:.3f}  "
                f"| feature: var={v}, mode={m+1}, lag={lag}"
            )

    # ============================================================
    # 4) Plot spike curve
    # ============================================================
    print("▶ Saving spike plot")

    plt.figure(figsize=(9, 4))
    plt.plot(
        df_curve["n_features"],
        df_curve["post_r2"],
        marker="o",
        linewidth=1.2,
        color="black"
    )

    plt.axhline(0, ls="--", linewidth=1, color="gray")

    for (n, dr2, r2) in spikes:
        plt.scatter(n, r2, color="orange", s=40, zorder=3)

    top10 = sorted(spikes, key=lambda x: x[1], reverse=True)[:10]
    for (n, dr2, r2) in top10:
        plt.scatter(n, r2, color="red", s=30, zorder=4)
        plt.text(
            n,
            r2 + 0.015,
            f"n={n}",
            color="red",
            ha="center",
            va="bottom",
            fontsize=8
        )

    title_suffix = "ensmean" if mode == "ensmean" else member_id
    plt.title(f"{MODEL} {TARGET} | Surface EOF spike curve (POST) | {title_suffix}")
    plt.xlabel("Number of ranked features")
    plt.ylabel("POST R²")
    plt.grid(True, linewidth=0.3)
    plt.tight_layout()

    plt.savefig(os.path.join(outdir, "spike_plot_post.png"), dpi=200)
    plt.savefig(os.path.join(outdir, "spike_plot_post.pdf"), dpi=200)
    plt.close()

    print(f"✅ Finished run. Outputs saved in:\n{outdir}")

# ============================================================
# Main
# ============================================================

if __name__ == "__main__":
    print(f"\n=== MODEL={MODEL} | TARGET={TARGET} | MODE={MODE} ===")
    print(f"EOF_DIR={EOF_DIR}")
    print(f"AMOC_FILE={AMOC_FILE}")

    if MODE == "ensmean":
        run_feature_workflow(mode="ensmean", member_id=None)

    elif MODE == "member":
        members = get_member_list()
        print("\nAvailable members:")
        print(members)

        if member_id is not None:
            if member_id not in members:
                raise ValueError(f"Chosen member_id '{member_id}' not found.")
            run_feature_workflow(mode="member", member_id=member_id)

        else:
            if not RUN_ALL_MEMBERS:
                raise ValueError(
                    "MODE='member' but member_id=None and RUN_ALL_MEMBERS=False. "
                    "Either choose a member_id or set RUN_ALL_MEMBERS=True."
                )

            for mem in members:
                run_feature_workflow(mode="member", member_id=mem)

    else:
        raise ValueError("MODE must be 'ensmean' or 'member'")

    print("\n✅ All requested runs completed.")


=== MODEL=MPI-ESM1-2-LR | TARGET=AMOC_45N_ensmean | MODE=ensmean ===
EOF_DIR=/data/projects/nckf/frekle/EOF_results/MPI-ESM1-2-LR/surface_latlon/Train_period_85pct/
AMOC_FILE=/data/users/frekle/AMOC_analysis/AMOC_MPI-ESM1-2-LR.nc

RUNNING FEATURE WORKFLOW | MODEL=MPI-ESM1-2-LR | TARGET=AMOC_45N_ensmean | MODE=ensmean | member=None
PRE ends at: 1989
PRE : 1850 - 1989 (140 years)
POST: 1990 - 2014 (25 years)
▶ Building PRE correlation ranking
✅ Saved: /data/users/frekle/Final_figures/MPI-ESM1-2-LR/AMOC_45N_ensmean/Feature_selection_surface/ensmean/feature_ranking_pre.csv
▶ Selecting alpha on POST by TimeSeriesSplit CV
Best alpha (POST CV): 4.0
▶ Building POST spike curve

Top spikes (POST):
n=  3  ΔR²=+0.147  R²=0.708  | feature: var=so, mode=8, lag=1
n=  4  ΔR²=+0.065  R²=0.772  | feature: var=so, mode=8, lag=2
n=  5  ΔR²=+0.048  R²=0.820  | feature: var=so, mode=4, lag=0
n= 14  ΔR²=+0.038  R²=0.908  | feature: var=so, mode=4, lag=5
n= 19  ΔR²=+0.023  R²=0.940  | feature: var=so, mode=

In [1]:
#!/usr/bin/env python3
"""
SPIKE-FEATURE SUBSET SEARCH (TRAIN-ONLY SURFACE EOF basis) — CURVE-BASED CANDIDATES
====================================================================================
Supports:
- MODE = "ensmean"  -> use ensemble-mean PCs and ensemble-mean AMOC
- MODE = "member"   -> use one chosen member, or all members automatically

Goal:
  Instead of using "top10 spikes txt", choose candidate features from
  spike_curve_post.csv in a consistent way.

Key idea:
  - A spike curve step n corresponds to "top-n ranked features".
  - We take candidate indices from the curve file, map n -> feature at rank n,
    and then do the same subset search among those candidates.

Outputs (per mode/member):
  - best_subsets_by_k.csv
  - all_subsets_scored.csv (optional)
  - summary_best_subsets.txt
  - spikes_top10_post.csv
  - spikes_top10_post.txt
"""

import os
import glob
import itertools
import numpy as np
import pandas as pd
import xarray as xr

from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import Ridge
from sklearn.metrics import r2_score

os.environ["HDF5_USE_FILE_LOCKING"] = "FALSE"

# ------------------------------------------------------------
# USER SETTINGS
# ------------------------------------------------------------
MODEL  = "CESM2"   # "EC-Earth3", "IPSL-CM6A-LR", "CESM2", "MPI-ESM1-2-LR"

# Examples:
#   ensmean mode: TARGET="AMOC_45N_ensmean"
#   member mode : TARGET="AMOC_45N_member"
TARGET = "AMOC_45N_ensmean"

# "ensmean" or "member"
MODE = "ensmean"

# If MODE="member":
#   choose one specific member, e.g. "r1i1p1f1"
#   or leave as None and set RUN_ALL_MEMBERS=True
member_id = None
RUN_ALL_MEMBERS = True

# Choose which AMOC you want to reconstruct:
#   "normal" -> uses TARGET as-is
#   "smooth" -> maps TARGET to AMOC_26N_smooth / AMOC_45N_smooth
AMOC_VARIANT = "normal"   # "normal" or "smooth"

EOF_DIR   = f"/data/projects/nckf/frekle/EOF_results/{MODEL}/surface_latlon/Train_period_85pct/"
AMOC_FILE = f"/data/users/frekle/AMOC_analysis/AMOC_{MODEL}.nc"

# Base folders from the surface feature-correlation workflow
FEATURE_BASE = f"/data/users/frekle/Final_figures/{MODEL}/{TARGET}/Feature_selection_surface/"

# Candidate choice
CANDIDATE_METHOD = "all"   # "top_delta", "top_r2", "all"
TOP_N_CANDIDATES = 10
MIN_N = 1
MAX_N = 60

# Subset selection criterion
SELECT_ON = "train"   # "train" or "test"
TIEBREAK_ON = "train"    # "train" or "test"

# ------------------------------------------------------------
# Precursor constraint: only allow features with lag >= MIN_LAG
# ------------------------------------------------------------
MIN_LAG = 4
MAX_LAG = None   # e.g. 20, or keep None

# Window choice:
#   "maxlag" -> start at GLOBAL_MAXLAG of remaining candidates
#   "minlag" -> start at MIN_LAG
WINDOW_LAG_POLICY = "minlag"   # "maxlag" or "minlag"

# PC setup
N_MODES = 10

# Ridge hyperparams
ALPHA = 1.0

# Detrending (TRAIN-fit)
DETREND_Y = False
DETREND_X = False

# Subset search settings
MAX_FEATURES_TO_USE = 3
SAVE_ALL_SUBSETS = True

# ------------------------------------------------------------
# Helpers
# ------------------------------------------------------------
def extract_years(time_coord):
    try:
        return xr.DataArray(time_coord).dt.year.values.astype(int)
    except Exception:
        t = np.asarray(time_coord)
        return np.array([int(str(x)[:4]) for x in t], dtype=int)

def get_example_eof_file():
    f = os.path.join(EOF_DIR, "EOF_surface_latlon_thetao.nc")
    if os.path.exists(f):
        return f
    hits = sorted(glob.glob(os.path.join(EOF_DIR, "EOF_surface_latlon_*.nc")))
    if not hits:
        raise FileNotFoundError(f"No EOF files found in {EOF_DIR}")
    return hits[0]

def get_member_list_from_eof():
    f = get_example_eof_file()
    ds = xr.open_dataset(f)

    pc_name = "PC" if "PC" in ds.data_vars else "pcs" if "pcs" in ds.data_vars else None
    if pc_name is None:
        ds.close()
        raise KeyError(f"No PC variable found in {f}. Expected 'PC' or 'pcs'.")

    if "member" not in ds[pc_name].dims:
        ds.close()
        raise ValueError("EOF file has no member dimension.")

    members = [str(m) for m in ds["member"].values]
    ds.close()
    return members

def load_pc_surface(var, n_modes, eof_dir, mode="ensmean", member_id=None):
    f = os.path.join(eof_dir, f"EOF_surface_latlon_{var}.nc")
    if not os.path.exists(f):
        raise FileNotFoundError(f"Missing EOF file: {f}")

    ds = xr.open_dataset(f)

    pc_name = "PC" if "PC" in ds.data_vars else "pcs" if "pcs" in ds.data_vars else None
    if pc_name is None:
        ds.close()
        raise KeyError(f"No PC variable found in {f}. Expected 'PC' or 'pcs'.")

    PC = ds[pc_name].isel(mode=slice(0, int(n_modes)))

    if "member" in PC.dims:
        if mode == "ensmean":
            PC = PC.mean("member")
        elif mode == "member":
            if member_id is None:
                ds.close()
                raise ValueError("member_id must be provided when mode='member'")
            member_vals = [str(m) for m in PC["member"].values]
            if member_id not in member_vals:
                ds.close()
                raise ValueError(f"member_id '{member_id}' not found in {f}")
            PC = PC.sel(member=member_id)
        else:
            ds.close()
            raise ValueError("mode must be 'ensmean' or 'member'")

    PC = PC.transpose("time", "mode")
    years = extract_years(PC["time"])
    PC = PC.assign_coords(year=("time", years)).swap_dims({"time": "year"}).drop_vars("time")
    ds.close()
    return PC.astype(float)

def _infer_amoc_lat_from_target(target_name: str):
    if "26N" in target_name:
        return "26N"
    if "45N" in target_name:
        return "45N"
    raise ValueError(f"Could not infer latitude tag (26N/45N) from TARGET='{target_name}'")

def resolve_amoc_variable(target: str, amoc_variant: str):
    amoc_variant = str(amoc_variant).strip().lower()
    if amoc_variant == "normal":
        return target
    if amoc_variant == "smooth":
        lat = _infer_amoc_lat_from_target(target)
        return f"AMOC_{lat}_smooth"
    raise ValueError("AMOC_VARIANT must be 'normal' or 'smooth'")

def load_amoc(target, amoc_file, mode="ensmean", member_id=None, amoc_variant="normal"):
    """
    Loads AMOC series.

    - normal:
        uses TARGET as-is (e.g. AMOC_45N_ensmean or AMOC_45N_member)
    - smooth:
        uses AMOC_26N_smooth / AMOC_45N_smooth

    For member mode:
    - if AMOC file has labeled member coords, selects directly
    - otherwise maps member_id onto numeric member index using EOF member ordering
    """
    varname = resolve_amoc_variable(target, amoc_variant)

    ds = xr.open_dataset(amoc_file)
    if varname not in ds:
        ds.close()
        raise KeyError(f"AMOC var '{varname}' not found in {amoc_file}. Available: {list(ds.data_vars)}")

    y = ds[varname].squeeze()

    if "member" in y.dims:
        if mode == "ensmean":
            y = y.mean("member")

        elif mode == "member":
            if member_id is None:
                ds.close()
                raise ValueError("member_id must be provided when mode='member'")

            eof_members = get_member_list_from_eof()

            if "member" in y.coords:
                member_vals = [str(m) for m in y["member"].values]
                if member_id in member_vals:
                    y = y.sel(member=member_id)
                else:
                    if member_id not in eof_members:
                        ds.close()
                        raise ValueError(f"member_id '{member_id}' not found in EOF member list")
                    idx = eof_members.index(member_id)
                    if idx >= y.sizes["member"]:
                        ds.close()
                        raise ValueError(f"Member index {idx} out of bounds for AMOC file")
                    y = y.isel(member=idx)
            else:
                if member_id not in eof_members:
                    ds.close()
                    raise ValueError(f"member_id '{member_id}' not found in EOF member list")
                idx = eof_members.index(member_id)
                if idx >= y.sizes["member"]:
                    ds.close()
                    raise ValueError(f"Member index {idx} out of bounds for AMOC file")
                y = y.isel(member=idx)
        else:
            ds.close()
            raise ValueError("mode must be 'ensmean' or 'member'")

    y = y.squeeze(drop=True)

    if "year" in y.dims:
        am = y
    else:
        years = extract_years(y["time"])
        am = y.assign_coords(year=("time", years)).swap_dims({"time": "year"}).drop_vars("time")

    ds.close()
    return am.astype(float).squeeze()

def infer_train_end_year_from_any_eof(eof_dir, prefer="so"):
    f = os.path.join(eof_dir, f"EOF_surface_latlon_{prefer}.nc")
    if not os.path.exists(f):
        hits = sorted(glob.glob(os.path.join(eof_dir, "EOF_surface_latlon_*.nc")))
        if not hits:
            raise FileNotFoundError(f"No EOF files found in {eof_dir}")
        f = hits[0]

    ds = xr.open_dataset(f)
    if "TRAIN_END" in ds.attrs:
        y = int(str(ds.attrs["TRAIN_END"])[:4])
        ds.close()
        return y
    if "train_mask" in ds:
        tm = ds["train_mask"].values.astype(bool)
        if tm.any():
            t_last = ds["time"].values[np.where(tm)[0][-1]]
            ds.close()
            return int(str(np.datetime64(t_last))[:4])
    ds.close()
    raise RuntimeError(f"Could not infer TRAIN_END year from {f}")

def fit_linear_trend(train_years, train_series):
    x = np.asarray(train_years, float)
    y = np.asarray(train_series, float)
    m = np.isfinite(x) & np.isfinite(y)
    x = x[m]
    y = y[m]
    if len(x) < 2:
        return 0.0, float(np.nanmean(y))
    A = np.vstack([x, np.ones_like(x)]).T
    a, b = np.linalg.lstsq(A, y, rcond=None)[0]
    return float(a), float(b)

def detrend_with_train_fit(all_years, all_series, train_mask):
    all_years = np.asarray(all_years, float)
    all_series = np.asarray(all_series, float)
    a, b = fit_linear_trend(all_years[train_mask], all_series[train_mask])
    trend = a * all_years + b
    return all_series - trend, (a, b)

def pearson_corr(a, b):
    a = np.asarray(a, float)
    b = np.asarray(b, float)
    m = np.isfinite(a) & np.isfinite(b)
    if m.sum() < 2:
        return np.nan
    a = a[m]
    b = b[m]
    sa = a.std()
    sb = b.std()
    if sa == 0 or sb == 0:
        return np.nan
    return float(np.corrcoef(a, b)[0, 1])

def load_ranked_features(csv_path):
    """
    Surface version.
    Expects: var, mode, lag
    mode is 1-based -> convert to 0-based.
    """
    df = pd.read_csv(csv_path)

    for c in ["var", "mode", "lag"]:
        if c not in df.columns:
            raise KeyError(f"Missing column '{c}' in {csv_path}. Columns: {list(df.columns)}")

    df["var"] = df["var"].astype(str).str.strip()
    df["mode0"] = df["mode"].astype(int) - 1
    df["lag"] = df["lag"].astype(int)

    feats = [(r["var"], int(r["mode0"]), int(r["lag"])) for _, r in df.iterrows()]
    return feats, df

def pick_candidate_ns_from_curve(curve_csv, method="top_delta", topK=10, min_n=1, max_n=None):
    df = pd.read_csv(curve_csv)

    n_col = None
    for c in ["n_features", "n", "k"]:
        if c in df.columns:
            n_col = c
            break
    if n_col is None:
        raise KeyError(f"Could not find an n column in {curve_csv}. Columns: {list(df.columns)}")

    df = df.copy()
    df[n_col] = pd.to_numeric(df[n_col], errors="coerce")
    df = df.dropna(subset=[n_col])

    df = df[df[n_col] >= int(min_n)]
    if max_n is not None:
        df = df[df[n_col] <= int(max_n)]

    if len(df) == 0:
        raise RuntimeError("Curve file has no rows after filtering MIN_N/MAX_N.")

    if method == "all":
        ns = df[n_col].astype(int).tolist()
        return sorted(set(ns))

    if method == "top_delta":
        score_col = None
        for c in ["delta_post_r2", "delta_r2", "delta_test_r2", "delta_r2_test"]:
            if c in df.columns:
                score_col = c
                break
        if score_col is None:
            raise KeyError(f"method='top_delta' but no delta column found. Columns: {list(df.columns)}")
        df[score_col] = pd.to_numeric(df[score_col], errors="coerce")
        df = df.dropna(subset=[score_col]).sort_values(score_col, ascending=False)

    elif method == "top_r2":
        score_col = None
        for c in ["post_r2", "test_r2", "r2", "r2_test"]:
            if c in df.columns:
                score_col = c
                break
        if score_col is None:
            raise KeyError(f"method='top_r2' but no r2 column found. Columns: {list(df.columns)}")
        df[score_col] = pd.to_numeric(df[score_col], errors="coerce")
        df = df.dropna(subset=[score_col]).sort_values(score_col, ascending=False)

    else:
        raise ValueError("CANDIDATE_METHOD must be one of: 'top_delta','top_r2','all'")

    df = df.head(int(topK))
    ns = df[n_col].astype(int).tolist()
    return sorted(set(ns))

def choose_better(curr, best, select_on="train", tiebreak_on="test"):
    if select_on not in ("train", "test"):
        raise ValueError("SELECT_ON must be 'train' or 'test'")
    if tiebreak_on not in ("train", "test"):
        raise ValueError("TIEBREAK_ON must be 'train' or 'test'")

    key_main = "r2_train" if select_on == "train" else "r2_test"
    key_tie  = "r2_train" if tiebreak_on == "train" else "r2_test"

    if curr[key_main] > best[key_main]:
        return True
    if np.isclose(curr[key_main], best[key_main]) and curr[key_tie] > best[key_tie]:
        return True
    return False

def finite_rows_mask(Y, X):
    Y = np.asarray(Y, float)
    X = np.asarray(X, float)
    m = np.isfinite(Y)
    if X.ndim == 1:
        m = m & np.isfinite(X)
    else:
        m = m & np.all(np.isfinite(X), axis=1)
    return m

def safe_r2(y_true, y_pred):
    y_true = np.asarray(y_true, float)
    y_pred = np.asarray(y_pred, float)
    if len(y_true) < 2:
        return np.nan
    return float(r2_score(y_true, y_pred))

def filter_feats_by_lag(feats, min_lag=0, max_lag=None):
    out = []
    for (v, m0, lag) in feats:
        if lag < int(min_lag):
            continue
        if max_lag is not None and lag > int(max_lag):
            continue
        out.append((v, m0, lag))
    return out

def feat_to_str(ft):
    v, m0, lag = ft
    return f"{v} EOF{int(m0)+1} lag{int(lag)}"

def get_feature_input_dir(mode, member_id=None):
    if mode == "ensmean":
        d = os.path.join(FEATURE_BASE, "ensmean")
    elif mode == "member":
        if member_id is None:
            raise ValueError("member_id must be provided when mode='member'")
        d = os.path.join(FEATURE_BASE, member_id)
    else:
        raise ValueError("mode must be 'ensmean' or 'member'")
    return d

def get_subset_output_dir(mode, member_id=None):
    base_input = get_feature_input_dir(mode, member_id)
    outdir = os.path.join(
        base_input,
        "Not_detrended",
        f"Selected_on_{SELECT_ON}",
        f"amoc_variant_{AMOC_VARIANT}",
        f"lag_policy_{MIN_LAG}minlag_{MAX_LAG if MAX_LAG is not None else 'None'}max"
    )
    os.makedirs(outdir, exist_ok=True)
    return outdir

def write_spikes_top10_post(
    curve_csv,
    rank_csv,
    outdir,
    candidate_method="top_delta",
    topK=10,
    min_n=1,
    max_n=None,
    mode_label="ensmean",
):
    dfc = pd.read_csv(curve_csv)

    n_col = None
    for c in ["n_features", "n", "k"]:
        if c in dfc.columns:
            n_col = c
            break
    if n_col is None:
        raise KeyError(f"No n column found in {curve_csv}. Columns={list(dfc.columns)}")

    dfc = dfc.copy()
    dfc[n_col] = pd.to_numeric(dfc[n_col], errors="coerce")
    dfc = dfc.dropna(subset=[n_col])
    dfc = dfc[dfc[n_col] >= int(min_n)]
    if max_n is not None:
        dfc = dfc[dfc[n_col] <= int(max_n)]

    if len(dfc) == 0:
        raise RuntimeError("Curve file empty after MIN_N/MAX_N filtering.")

    score_col = None
    if candidate_method == "top_delta":
        for c in ["delta_post_r2", "delta_r2", "delta_test_r2", "delta_r2_test"]:
            if c in dfc.columns:
                score_col = c
                break
        if score_col is None:
            raise KeyError(f"candidate_method='top_delta' but no delta column in {curve_csv}.")
        dfc[score_col] = pd.to_numeric(dfc[score_col], errors="coerce")
        dfc = dfc.dropna(subset=[score_col]).sort_values(score_col, ascending=False)

    elif candidate_method == "top_r2":
        for c in ["post_r2", "test_r2", "r2", "r2_test"]:
            if c in dfc.columns:
                score_col = c
                break
        if score_col is None:
            raise KeyError(f"candidate_method='top_r2' but no r2 column in {curve_csv}.")
        dfc[score_col] = pd.to_numeric(dfc[score_col], errors="coerce")
        dfc = dfc.dropna(subset=[score_col]).sort_values(score_col, ascending=False)

    elif candidate_method == "all":
        score_col = None
        dfc = dfc.sort_values(n_col, ascending=True)

    else:
        raise ValueError("candidate_method must be 'top_delta', 'top_r2', or 'all'.")

    dfc_top = dfc.head(int(topK)).copy()
    ns = dfc_top[n_col].astype(int).tolist()

    feats_ranked, _ = load_ranked_features(rank_csv)

    rows = []
    for n in ns:
        if n < 1 or n > len(feats_ranked):
            continue
        ft = feats_ranked[n - 1]
        row = {"n": int(n), "feature": feat_to_str(ft)}
        if score_col is not None and score_col in dfc_top.columns:
            row["score"] = float(dfc_top.loc[dfc_top[n_col].astype(int) == n, score_col].iloc[0])
        else:
            row["score"] = np.nan

        for c in ["post_r2", "test_r2", "r2", "r2_test"]:
            if c in dfc_top.columns:
                row["curve_r2"] = float(dfc_top.loc[dfc_top[n_col].astype(int) == n, c].iloc[0])
                break
        if "curve_r2" not in row:
            row["curve_r2"] = np.nan

        rows.append(row)

    if len(rows) == 0:
        raise RuntimeError("No valid n entries to write spikes_top10_post (mapping failed?).")

    df_out = pd.DataFrame(rows)

    out_csv = os.path.join(outdir, "spikes_top10_post.csv")
    df_out.to_csv(out_csv, index=False)
    print("✅ Wrote:", out_csv)

    out_txt = os.path.join(outdir, "spikes_top10_post.txt")
    with open(out_txt, "w") as fp:
        fp.write("Top curve-selected candidate steps (mapped to ranked features)\n")
        fp.write(f"MODEL={MODEL}  TARGET={TARGET}  MODE={MODE}  RUN={mode_label}\n")
        fp.write(f"curve_file={curve_csv}\n")
        fp.write(f"candidate_method={candidate_method}  topK={topK}  score_col={score_col}\n")
        fp.write("\n")
        for _, r in df_out.iterrows():
            n = int(r["n"])
            feat = str(r["feature"])
            score = r["score"]
            cr2 = r["curve_r2"]
            if np.isfinite(score):
                fp.write(f"n={n:3d}  score={score:+.4f}  curve_r2={cr2:.4f}  feature={feat}\n")
            else:
                fp.write(f"n={n:3d}  curve_r2={cr2:.4f}  feature={feat}\n")

    print("✅ Wrote:", out_txt)
    return out_txt, out_csv

# ------------------------------------------------------------
# Main workflow for one mode/member
# ------------------------------------------------------------
def run_subset_search(mode="ensmean", member_id=None):
    run_label = "ensmean" if mode == "ensmean" else str(member_id)
    print("\n" + "=" * 90)
    print(f"RUNNING SUBSET SEARCH | MODEL={MODEL} | TARGET={TARGET} | MODE={mode} | RUN={run_label}")
    print("=" * 90)

    in_dir = get_feature_input_dir(mode, member_id)
    outdir = get_subset_output_dir(mode, member_id)

    rank_csv = os.path.join(in_dir, "feature_ranking_pre.csv")
    spike_curve_csv = os.path.join(in_dir, "spike_curve_post.csv")

    if not os.path.exists(rank_csv):
        raise FileNotFoundError(f"Missing ranking file: {rank_csv}")
    if not os.path.exists(spike_curve_csv):
        raise FileNotFoundError(f"Missing spike curve file: {spike_curve_csv}")

    # ------------------------------------------------------------
    # 1) Build curve-based candidate feature list
    # ------------------------------------------------------------
    feats_ranked, _ = load_ranked_features(rank_csv)

    curve_ns = pick_candidate_ns_from_curve(
        spike_curve_csv,
        method=CANDIDATE_METHOD,
        topK=TOP_N_CANDIDATES,
        min_n=MIN_N,
        max_n=MAX_N
    )

    cands = []
    for n in curve_ns:
        if n < 1 or n > len(feats_ranked):
            print(f"⚠ curve n={n} outside ranked list length={len(feats_ranked)} -> skipping")
            continue
        cands.append(feats_ranked[n - 1])

    seen = set()
    cand_feats = []
    for ft in cands:
        if ft not in seen:
            cand_feats.append(ft)
            seen.add(ft)

    cand_feats = filter_feats_by_lag(cand_feats, min_lag=MIN_LAG, max_lag=MAX_LAG)

    M = len(cand_feats)
    if M == 0:
        raise RuntimeError(
            f"No candidate features left after lag filtering: MIN_LAG={MIN_LAG}, MAX_LAG={MAX_LAG}"
        )

    print("\n==============================")
    print("CURVE-BASED CANDIDATE FEATURES")
    print("==============================")
    print(f"Curve ns used: {curve_ns}")
    print(f"Candidate features (unique): {M}")
    print(f"MIN_LAG filter: >= {MIN_LAG}" + (f", <= {MAX_LAG}" if MAX_LAG is not None else ""))
    for i, (v, m0, lag) in enumerate(cand_feats, 1):
        print(f"  C{i}: {v} EOF{m0+1} lag{lag}")

    KMAX = min(MAX_FEATURES_TO_USE, M)

    # ------------------------------------------------------------
    # 2) Load AMOC + PCs needed
    # ------------------------------------------------------------
    TRAIN_END_YEAR = infer_train_end_year_from_any_eof(EOF_DIR, prefer="so")
    print("\n✅ TRAIN_END_YEAR inferred from EOF files:", TRAIN_END_YEAR)

    amoc_var_used = resolve_amoc_variable(TARGET, AMOC_VARIANT)
    print(f"✅ AMOC variable used: {amoc_var_used}   (AMOC_VARIANT={AMOC_VARIANT})")

    amoc = load_amoc(TARGET, AMOC_FILE, mode=mode, member_id=member_id, amoc_variant=AMOC_VARIANT)

    needed_vars = sorted(set(v for (v, _, _) in cand_feats))
    pc_dict = {
        v: load_pc_surface(v, N_MODES, EOF_DIR, mode=mode, member_id=member_id)
        for v in needed_vars
    }

    common_years = amoc["year"].values.astype(int)
    for da in pc_dict.values():
        common_years = np.intersect1d(common_years, da["year"].values.astype(int))
    years = np.asarray(common_years, int)
    years.sort()

    y_amoc = amoc.sel(year=years).values.astype(float)
    pc_np = {k: v.sel(year=years).values.astype(float) for k, v in pc_dict.items()}

    # ------------------------------------------------------------
    # 3) Fix evaluation window (GLOBAL_MAXLAG) for fair comparison
    # ------------------------------------------------------------
    GLOBAL_MAXLAG = max(lag for *_, lag in cand_feats)

    if WINDOW_LAG_POLICY == "maxlag":
        START_LAG = GLOBAL_MAXLAG
    elif WINDOW_LAG_POLICY == "minlag":
        START_LAG = int(MIN_LAG)
    else:
        raise ValueError("WINDOW_LAG_POLICY must be 'maxlag' or 'minlag'")

    used_global = np.arange(START_LAG, len(years), dtype=int)

    years_used = years[used_global]
    Y_all = y_amoc[used_global]

    idx_tr = np.where(years_used <= TRAIN_END_YEAR)[0]
    idx_te = np.where(years_used > TRAIN_END_YEAR)[0]

    if len(idx_tr) == 0 or len(idx_te) == 0:
        raise RuntimeError("Train/test split became empty after lag windowing.")

    print("\nSplit on years_used (fixed by GLOBAL_MAXLAG):")
    print("GLOBAL_MAXLAG (of candidates):", GLOBAL_MAXLAG)
    print("START_LAG (window starts):", START_LAG, f"(policy={WINDOW_LAG_POLICY})")
    print("Train:", years_used[idx_tr[0]], "–", years_used[idx_tr[-1]], "n=", len(idx_tr))
    print("Test :", years_used[idx_te[0]], "–", years_used[idx_te[-1]], "n=", len(idx_te))

    train_mask_used = np.zeros(len(years_used), dtype=bool)
    train_mask_used[idx_tr] = True

    if DETREND_Y:
        Y_dt, (ay, by) = detrend_with_train_fit(years_used, Y_all, train_mask_used)
        print(f"✅ Detrended Y using TRAIN fit: y_trend = {ay:.4e}*year + {by:.4e}")
    else:
        Y_dt = Y_all.copy()

    X_cand = np.empty((len(used_global), M), dtype=float)
    for j, (var, m0, lag) in enumerate(cand_feats):
        X_cand[:, j] = pc_np[var][used_global - lag, m0]

    if DETREND_X:
        X_cand_dt = X_cand.copy()
        for j in range(M):
            X_cand_dt[:, j], _ = detrend_with_train_fit(years_used, X_cand[:, j], train_mask_used)
    else:
        X_cand_dt = X_cand

    # ------------------------------------------------------------
    # 4) Exhaustive subset search among curve candidates
    # ------------------------------------------------------------
    all_rows = []
    best_by_k = []

    print("\n==============================")
    print("SUBSET SEARCH")
    print("==============================")
    print(f"Selecting BEST subsets by: {SELECT_ON.upper()} R² (tie-break: {TIEBREAK_ON.upper()} R²)")
    print(f"Target AMOC: {amoc_var_used}")
    print(f"ALPHA={ALPHA}  DETREND_Y={DETREND_Y}  DETREND_X={DETREND_X}")
    print(f"KMAX={KMAX}  Candidates={M}\n")

    for k in range(1, KMAX + 1):
        best = {
            "k": k,
            "r2_train": -np.inf,
            "corr_train": np.nan,
            "r2_test": -np.inf,
            "corr_test": np.nan,
            "subset_idx": None,
        }

        for subset_idx in itertools.combinations(range(M), k):
            subset_idx = tuple(subset_idx)
            Xk = X_cand_dt[:, subset_idx]

            mdl = make_pipeline(StandardScaler(), Ridge(alpha=float(ALPHA)))

            Ytr = Y_dt[idx_tr]
            Xtr = Xk[idx_tr]
            m_tr = finite_rows_mask(Ytr, Xtr)

            Yte = Y_dt[idx_te]
            Xte = Xk[idx_te]
            m_te = finite_rows_mask(Yte, Xte)

            if m_tr.sum() < 3:
                continue
            if m_te.sum() < 2:
                continue

            mdl.fit(Xtr[m_tr], Ytr[m_tr])

            pred_tr = mdl.predict(Xtr[m_tr])
            r2_tr = safe_r2(Ytr[m_tr], pred_tr)
            c_tr = pearson_corr(Ytr[m_tr], pred_tr)

            pred_te = mdl.predict(Xte[m_te])
            r2_te = safe_r2(Yte[m_te], pred_te)
            c_te = pearson_corr(Yte[m_te], pred_te)

            if not np.isfinite(r2_tr) or not np.isfinite(r2_te):
                continue

            curr = {
                "r2_train": r2_tr,
                "corr_train": c_tr,
                "r2_test": r2_te,
                "corr_test": c_te,
                "subset_idx": subset_idx,
            }

            if SAVE_ALL_SUBSETS:
                subset_feats = [cand_feats[i] for i in subset_idx]
                subset_str = " | ".join([f"{v} EOF{m0+1} lag{lag}" for (v, m0, lag) in subset_feats])
                all_rows.append({
                    "k": k,
                    "r2_train": r2_tr,
                    "corr_train": c_tr,
                    "r2_test": r2_te,
                    "corr_test": c_te,
                    "subset_idx": ",".join(map(str, subset_idx)),
                    "subset_features": subset_str,
                    "candidate_source": os.path.basename(spike_curve_csv),
                    "candidate_method": CANDIDATE_METHOD,
                    "selected_by": SELECT_ON,
                    "amoc_variant": AMOC_VARIANT,
                    "amoc_var_used": amoc_var_used,
                    "run_label": run_label,
                })

            if best["subset_idx"] is None or choose_better(curr, best, select_on=SELECT_ON, tiebreak_on=TIEBREAK_ON):
                best["r2_train"] = r2_tr
                best["corr_train"] = c_tr
                best["r2_test"] = r2_te
                best["corr_test"] = c_te
                best["subset_idx"] = subset_idx

        if best["subset_idx"] is None:
            raise RuntimeError(f"No valid subset found for k={k} in run {run_label}")

        subset_feats = [cand_feats[i] for i in best["subset_idx"]]
        subset_str = " | ".join([f"{v} EOF{m0+1} lag{lag}" for (v, m0, lag) in subset_feats])

        best_by_k.append({
            "k": k,
            "r2_train": best["r2_train"],
            "corr_train": best["corr_train"],
            "r2_test": best["r2_test"],
            "corr_test": best["corr_test"],
            "subset_idx": ",".join(map(str, best["subset_idx"])),
            "subset_features": subset_str,
            "candidate_source": os.path.basename(spike_curve_csv),
            "candidate_method": CANDIDATE_METHOD,
            "selected_by": SELECT_ON,
            "amoc_variant": AMOC_VARIANT,
            "amoc_var_used": amoc_var_used,
            "run_label": run_label,
        })

        print(
            f"BEST k={k} (selected on {SELECT_ON}): "
            f"Train R²={best['r2_train']:.4f} corr={best['corr_train']:.4f} | "
            f"Test R²={best['r2_test']:.4f} corr={best['corr_test']:.4f}"
        )
        for (v, m0, lag) in subset_feats:
            print(f"  - {v:6s} EOF{m0+1:<2d} lag{lag}")
        print("")

    # ------------------------------------------------------------
    # 5) Save results
    # ------------------------------------------------------------
    best_df = pd.DataFrame(best_by_k)
    best_csv = os.path.join(outdir, "best_subsets_by_k.csv")
    best_df.to_csv(best_csv, index=False)
    print("\n✅ Saved best subsets by k:", best_csv)

    if SAVE_ALL_SUBSETS:
        sort_main = "r2_train" if SELECT_ON == "train" else "r2_test"
        sort_tie  = "r2_train" if TIEBREAK_ON == "train" else "r2_test"
        all_df = (
            pd.DataFrame(all_rows)
            .sort_values(["k", sort_main, sort_tie], ascending=[True, False, False])
            .reset_index(drop=True)
        )
        all_csv = os.path.join(outdir, "all_subsets_scored.csv")
        all_df.to_csv(all_csv, index=False)
        print("✅ Saved all subset scores:", all_csv)

    summary_txt = os.path.join(outdir, "summary_best_subsets.txt")
    with open(summary_txt, "w") as f:
        f.write(f"MODEL={MODEL}\nTARGET={TARGET}\nMODE={MODE}\nRUN_LABEL={run_label}\n")
        f.write(f"member_id={member_id}\n")
        f.write(f"AMOC_VARIANT={AMOC_VARIANT}\nAMOC_VAR_USED={amoc_var_used}\n")
        f.write(f"TRAIN_END_YEAR={TRAIN_END_YEAR}\nGLOBAL_MAXLAG={GLOBAL_MAXLAG}\n")
        f.write(f"WINDOW_LAG_POLICY={WINDOW_LAG_POLICY}\nSTART_LAG={START_LAG}\n")
        f.write(f"MIN_LAG={MIN_LAG}\nMAX_LAG={MAX_LAG}\n")
        f.write(f"ALPHA={ALPHA}\nDETREND_Y={DETREND_Y}\nDETREND_X={DETREND_X}\n")
        f.write(f"SPIKE_CURVE_CSV={spike_curve_csv}\n")
        f.write(f"RANK_CSV={rank_csv}\n")
        f.write(f"CANDIDATE_METHOD={CANDIDATE_METHOD}\n")
        f.write(f"CURVE_NS={curve_ns}\n")
        f.write(f"SELECT_ON={SELECT_ON}\nTIEBREAK_ON={TIEBREAK_ON}\n")
        f.write("\nBest subsets by k:\n")
        for row in best_by_k:
            f.write(
                f"\nK={row['k']}  "
                f"R2_train={row['r2_train']:.6f}  corr_train={row['corr_train']:.6f}  "
                f"R2_test={row['r2_test']:.6f}  corr_test={row['corr_test']:.6f}\n"
            )
            f.write(row["subset_features"] + "\n")
    print("✅ Saved summary:", summary_txt)

    # ------------------------------------------------------------
    # 6) Write spikes_top10_post.txt / csv
    # ------------------------------------------------------------
    write_spikes_top10_post(
        curve_csv=spike_curve_csv,
        rank_csv=rank_csv,
        outdir=outdir,
        candidate_method=CANDIDATE_METHOD,
        topK=TOP_N_CANDIDATES,
        min_n=MIN_N,
        max_n=MAX_N,
        mode_label=run_label,
    )

    print("\n✅ DONE:", run_label)

# ------------------------------------------------------------
# Entry point
# ------------------------------------------------------------
if __name__ == "__main__":
    print(f"\n=== MODEL={MODEL} | TARGET={TARGET} | MODE={MODE} ===")
    print(f"EOF_DIR={EOF_DIR}")
    print(f"AMOC_FILE={AMOC_FILE}")
    print(f"FEATURE_BASE={FEATURE_BASE}")

    if MODE == "ensmean":
        run_subset_search(mode="ensmean", member_id=None)

    elif MODE == "member":
        members = get_member_list_from_eof()
        print("\nAvailable members:")
        print(members)

        if member_id is not None:
            if member_id not in members:
                raise ValueError(f"Chosen member_id '{member_id}' not found in EOF files.")
            run_subset_search(mode="member", member_id=member_id)

        else:
            if not RUN_ALL_MEMBERS:
                raise ValueError(
                    "MODE='member' but member_id=None and RUN_ALL_MEMBERS=False. "
                    "Either choose a member_id or set RUN_ALL_MEMBERS=True."
                )

            for mem in members:
                run_subset_search(mode="member", member_id=mem)

    else:
        raise ValueError("MODE must be 'ensmean' or 'member'")

    print("\n✅ All requested subset-search runs completed.")


=== MODEL=CESM2 | TARGET=AMOC_45N_ensmean | MODE=ensmean ===
EOF_DIR=/data/projects/nckf/frekle/EOF_results/CESM2/surface_latlon/Train_period_85pct/
AMOC_FILE=/data/users/frekle/AMOC_analysis/AMOC_CESM2.nc
FEATURE_BASE=/data/users/frekle/Final_figures/CESM2/AMOC_45N_ensmean/Feature_selection_surface/

RUNNING SUBSET SEARCH | MODEL=CESM2 | TARGET=AMOC_45N_ensmean | MODE=ensmean | RUN=ensmean

CURVE-BASED CANDIDATE FEATURES
Curve ns used: [1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20, 21, 22, 23, 24, 25, 26, 27, 28, 29, 30, 31, 32, 33, 34, 35, 36, 37, 38, 39, 40, 41, 42, 43, 44, 45, 46, 47, 48, 49, 50, 51, 52, 53, 54, 55, 56, 57, 58, 59, 60]
Candidate features (unique): 37
MIN_LAG filter: >= 4
  C1: thetao EOF3 lag10
  C2: thetao EOF3 lag11
  C3: thetao EOF3 lag8
  C4: thetao EOF3 lag9
  C5: thetao EOF3 lag7
  C6: thetao EOF3 lag12
  C7: thetao EOF3 lag5
  C8: thetao EOF3 lag4
  C9: so EOF3 lag4
  C10: thetao EOF3 lag6
  C11: thetao EOF3 lag13
  C12: thetao EOF3

/dmidata/users/frekle/miniforge3/envs/aimoc_env/lib/python3.11/site-packages/xarray/backends/plugins.py:80: RuntimeWarning: Engine 'gini' loading failed:
cannot import name 'cartopy_utils' from partially initialized module 'metpy.plots' (most likely due to a circular import) (/dmidata/users/frekle/miniforge3/envs/aimoc_env/lib/python3.11/site-packages/metpy/plots/__init__.py)
  warnings.warn(f"Engine {name!r} loading failed:\n{ex}", RuntimeWarning)



✅ TRAIN_END_YEAR inferred from EOF files: 1989
✅ AMOC variable used: AMOC_45N_ensmean   (AMOC_VARIANT=normal)

Split on years_used (fixed by GLOBAL_MAXLAG):
GLOBAL_MAXLAG (of candidates): 20
START_LAG (window starts): 4 (policy=minlag)
Train: 1854 – 1989 n= 136
Test : 1990 – 2014 n= 25

SUBSET SEARCH
Selecting BEST subsets by: TRAIN R² (tie-break: TRAIN R²)
Target AMOC: AMOC_45N_ensmean
ALPHA=1.0  DETREND_Y=False  DETREND_X=False
KMAX=3  Candidates=37

BEST k=1 (selected on train): Train R²=0.5727 corr=0.7568 | Test R²=-0.3072 corr=0.6977
  - thetao EOF3  lag11

BEST k=2 (selected on train): Train R²=0.6805 corr=0.8250 | Test R²=0.3292 corr=0.8752
  - thetao EOF3  lag12
  - thetao EOF3  lag4

BEST k=3 (selected on train): Train R²=0.7601 corr=0.8718 | Test R²=0.4394 corr=0.8812
  - thetao EOF3  lag11
  - thetao EOF3  lag4
  - so     EOF9  lag4


✅ Saved best subsets by k: /data/users/frekle/Final_figures/CESM2/AMOC_45N_ensmean/Feature_selection_surface/ensmean/Not_detrended/Selected_o

### Figures

In [2]:
#!/usr/bin/env python3
"""
FINAL FIGURES MASTER SCRIPT (SURFACE LAT-LON EOF workflow)
==========================================================
This script combines:
  (1) Reconstruction (true vs pred) for k=1..K_MAX (K_MAX<=5)
  (2) Mode panels: EOF(surface lat-lon) map + PC spaghetti/mean
      (up to 5 unique modes)
  (3) Rolling AR(1) plots: one per unique mode-set
  (4) Heatmap of |lag-correlation| for ALL spike modes from spikes_top10_post.txt
      (unique (var,mode), lags 0..LAG_MAX)
  (5) Contribution plot: how much each added feature changes TEST R²
      (waterfall ΔR²), based on chosen subset-search best features

Inputs expected (surface workflow):
  - feature_ranking_pre.csv
  - spikes_top10_post.txt
  - best_subsets_by_k.csv

Outputs:
  - 01_reconstruction_k1..kKMAX.(png/pdf)
  - 01_reconstruction_ALL_k1toKMAX.(png/pdf)
  - 02_EOFmap_and_PC_k....(png/pdf)
  - 03_AR1_k....(png/pdf)
  - 04_spike_modes_lagcorr_heatmap_top10.(png/pdf) + csv
  - 05_r2_contribution_waterfall_kmax.(png/pdf) + csv

Notes:
  - Split year inferred from EOF file TRAIN_END or train_mask.
  - Detrending uses TRAIN-fit only (no leakage) if enabled.
  - Alpha is fixed here (ALPHA_FIXED).
  - Reconstruction scoring uses TEST window (post TRAIN_END_YEAR).
"""

import os
import re
import glob
import numpy as np
import pandas as pd
import xarray as xr
import matplotlib.pyplot as plt

from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import Ridge
from sklearn.metrics import r2_score

from matplotlib.ticker import MultipleLocator
import matplotlib.colors as mcolors
import matplotlib.ticker as mticker

os.environ["HDF5_USE_FILE_LOCKING"] = "FALSE"

# ============================================================
# USER SETTINGS
# ============================================================
MODEL  = "CESM2"  # "EC-Earth3", "IPSL-CM6A-LR", "CESM2", "MPI-ESM1-2-LR"
TARGET = "AMOC_45N_ensmean"
MODE   = "ensmean"      # "ensmean" or "member"
member_id = None        # if MODE="member": label or integer index

EOF_DIR   = f"/data/projects/nckf/frekle/EOF_results/{MODEL}/surface_latlon/Train_period_85pct/"
AMOC_FILE = f"/data/users/frekle/AMOC_analysis/AMOC_{MODEL}.nc"

# Choose which AMOC series to PLOT/RECONSTRUCT as "truth"
#   "normal" -> uses TARGET as-is
#   "smooth" -> uses AMOC_26N_smooth or AMOC_45N_smooth
AMOC_VARIANT = "normal"   # "normal" or "smooth"

# Inputs produced by your surface pipeline
RANK_CSV   = f"/data/users/frekle/Final_figures/{MODEL}/{TARGET}/Feature_selection_surface/ensmean/feature_ranking_pre.csv"
SPIKES_TXT = f"/data/users/frekle/Final_figures/{MODEL}/{TARGET}/Feature_selection_surface/ensmean/Not_detrended/Selected_on_train/amoc_variant_{AMOC_VARIANT}/lag_policy_4minlag_Nonemax/spikes_top10_post.txt"
BEST_CSV   = f"/data/users/frekle/Final_figures/{MODEL}/{TARGET}/Feature_selection_surface/ensmean/Not_detrended/Selected_on_train/amoc_variant_{AMOC_VARIANT}/lag_policy_4minlag_Nonemax/best_subsets_by_k.csv"

# Output
OUTDIR = f"/data/users/frekle/Final_figures/{MODEL}/{TARGET}/Best_test_R2_surface/Not_detrended/Selected_on_train/amoc_variant_{AMOC_VARIANT}/minlag_4/"
os.makedirs(OUTDIR, exist_ok=True)

# Feature / plot settings
K_MAX = 5
MAX_MODE_PANELS = 5
N_MODES = 10
LAG_MAX = 20
CORR_ON = "train"            # "train" or "all" for heatmap

# Reconstruction preprocessing
DETREND_Y = False
DETREND_X = False

# Alpha (fixed; no tuning)
ALPHA_FIXED = 1.0

# AR(1)
AR1_WINDOW_YEARS = 30
AR1_MIN_VALID = 10

# Aesthetic controls
PC_SPAGHETTI_ALPHA = 0.25
PC_SPAGHETTI_LW = 0.6
PC_MEAN_LW = 1.8

# ============================================================
# HELPERS
# ============================================================
def savefig(outbase, dpi=220):
    plt.savefig(outbase + ".png", dpi=dpi, bbox_inches="tight")
    plt.savefig(outbase + ".pdf", dpi=dpi, bbox_inches="tight")

def extract_years(time_coord):
    try:
        return xr.DataArray(time_coord).dt.year.values.astype(int)
    except Exception:
        t = np.asarray(time_coord)
        return np.array([int(str(x)[:4]) for x in t], dtype=int)

def format_year_axis(ax, years_arr, step=10, pad=5):
    years_arr = np.asarray(years_arr, int)
    ax.set_xlim(int(years_arr.min()) - pad, int(years_arr.max()) + pad)
    ax.xaxis.set_major_locator(MultipleLocator(step))

def infer_train_end_year_from_any_eof(eof_dir, prefer="so"):
    f = os.path.join(eof_dir, f"EOF_surface_latlon_{prefer}.nc")
    if not os.path.exists(f):
        hits = sorted(glob.glob(os.path.join(eof_dir, "EOF_surface_latlon_*.nc")))
        if not hits:
            raise FileNotFoundError(f"No EOF files found in {eof_dir}")
        f = hits[0]

    ds = xr.open_dataset(f)
    if "TRAIN_END" in ds.attrs:
        y = int(str(ds.attrs["TRAIN_END"])[:4])
        ds.close()
        return y
    if "train_mask" in ds:
        tm = ds["train_mask"].values.astype(bool)
        if tm.any():
            t_last = ds["time"].values[np.where(tm)[0][-1]]
            ds.close()
            return int(str(np.datetime64(t_last))[:4])
    ds.close()
    raise RuntimeError(f"Could not infer TRAIN_END year from {f}")

def _infer_amoc_lat_from_target(target_name: str):
    if "26N" in target_name:
        return "26N"
    if "45N" in target_name:
        return "45N"
    raise ValueError(f"Could not infer latitude tag (26N/45N) from TARGET='{target_name}'")

def resolve_amoc_variable(target: str, amoc_variant: str):
    amoc_variant = str(amoc_variant).strip().lower()
    if amoc_variant == "normal":
        return target
    if amoc_variant == "smooth":
        lat = _infer_amoc_lat_from_target(target)
        return f"AMOC_{lat}_smooth"
    raise ValueError("AMOC_VARIANT must be 'normal' or 'smooth'")

def load_amoc(target, amoc_file, mode="ensmean", member_id=None, amoc_variant="normal"):
    varname = resolve_amoc_variable(target, amoc_variant)

    ds = xr.open_dataset(amoc_file)
    if varname not in ds.data_vars:
        ds.close()
        raise KeyError(f"AMOC var '{varname}' not found in {amoc_file}. Available: {list(ds.data_vars)}")

    y = ds[varname].squeeze()

    if "year" in y.dims:
        yy = y
        if "time" in yy.coords:
            yy = yy.drop_vars("time")
    else:
        years = extract_years(y["time"])
        yy = y.assign_coords(year=("time", years)).swap_dims({"time": "year"}).drop_vars("time")

    yy = yy.astype(float)

    if "member" in yy.dims:
        if mode == "ensmean":
            yy = yy.mean("member")
        elif mode == "member":
            if member_id is None:
                ds.close()
                raise ValueError("member_id must be provided when mode='member'")
            if "member" in yy.coords and member_id in [str(m) for m in yy["member"].values]:
                yy = yy.sel(member=member_id)
            else:
                yy = yy.isel(member=int(member_id))
        else:
            ds.close()
            raise ValueError("mode must be 'ensmean' or 'member'")

    ds.close()
    return yy.squeeze()

def _get_pc_varname(ds):
    if "PC" in ds.data_vars:
        return "PC"
    if "pcs" in ds.data_vars:
        return "pcs"
    raise KeyError(f"No PC variable found. Available: {list(ds.data_vars)}")

def load_pc_surface(var, n_modes, eof_dir, mode="ensmean", member_id=None):
    f = os.path.join(eof_dir, f"EOF_surface_latlon_{var}.nc")
    if not os.path.exists(f):
        raise FileNotFoundError(f"Missing EOF file: {f}")

    ds = xr.open_dataset(f)
    pc_name = _get_pc_varname(ds)
    PC = ds[pc_name].isel(mode=slice(0, int(n_modes)))

    if "member" in PC.dims:
        if mode == "ensmean":
            PC = PC.mean("member")
        else:
            if member_id is None:
                ds.close()
                raise ValueError("member_id must be provided when mode='member'")
            member_vals = [str(m) for m in PC["member"].values]
            if member_id in member_vals:
                PC = PC.sel(member=member_id)
            else:
                PC = PC.isel(member=int(member_id))

    PC = PC.transpose("time", "mode")
    years = extract_years(PC["time"])
    PC = PC.assign_coords(year=("time", years)).swap_dims({"time": "year"}).drop_vars("time")
    ds.close()
    return PC.astype(float)

def load_pc_surface_with_members(var, n_modes, eof_dir):
    f = os.path.join(eof_dir, f"EOF_surface_latlon_{var}.nc")
    if not os.path.exists(f):
        raise FileNotFoundError(f"Missing EOF file: {f}")

    ds = xr.open_dataset(f)
    pc_name = _get_pc_varname(ds)
    PC = ds[pc_name].isel(mode=slice(0, int(n_modes)))
    years = extract_years(PC["time"])
    PC = PC.assign_coords(year=("time", years)).swap_dims({"time":"year"}).drop_vars("time")
    ds.close()
    return PC.astype(float)

def load_eof_map_surface(var, eof_dir, mode_index_1based):
    f = os.path.join(eof_dir, f"EOF_surface_latlon_{var}.nc")
    if not os.path.exists(f):
        raise FileNotFoundError(f"Missing EOF file: {f}")

    ds = xr.open_dataset(f)

    eof_var_candidates = ["EOF", "eof", "patterns", "EOFs", "eofs_corr"]
    eof_name = next((c for c in eof_var_candidates if c in ds.data_vars), None)
    if eof_name is None:
        raise KeyError(f"Could not find EOF variable in {f}. Available: {list(ds.data_vars)}")

    EOF = ds[eof_name]
    m0 = int(mode_index_1based) - 1
    EOFm = EOF.isel(mode=m0)

    # expected dims: (y,x) or equivalent
    if EOFm.ndim != 2:
        raise ValueError(f"Expected 2D surface EOF, got dims={EOFm.dims}")

    # coordinates
    if "lat" in ds.coords:
        lat = ds["lat"].values.astype(float)
    elif "lat" in ds:
        lat = ds["lat"].values.astype(float)
    else:
        raise KeyError(f"No latitude field found in {f}")

    if "lon" in ds.coords:
        lon = ds["lon"].values.astype(float)
    elif "lon" in ds:
        lon = ds["lon"].values.astype(float)
    else:
        raise KeyError(f"No longitude field found in {f}")

    Z = EOFm.values.astype(float)

    ds.close()
    return Z, lat, lon

def fit_linear_trend(train_years, train_series):
    x = np.asarray(train_years, float)
    y = np.asarray(train_series, float)
    m = np.isfinite(x) & np.isfinite(y)
    x = x[m]
    y = y[m]
    if len(x) < 2:
        return 0.0, float(np.nanmean(y))
    A = np.vstack([x, np.ones_like(x)]).T
    a, b = np.linalg.lstsq(A, y, rcond=None)[0]
    return float(a), float(b)

def detrend_with_train_fit(all_years, all_series, train_mask):
    all_years = np.asarray(all_years, float)
    all_series = np.asarray(all_series, float)
    a, b = fit_linear_trend(all_years[train_mask], all_series[train_mask])
    return all_series - (a * all_years + b), (a, b)

def pearson_corr(a, b):
    a = np.asarray(a, float)
    b = np.asarray(b, float)
    m = np.isfinite(a) & np.isfinite(b)
    if m.sum() < 3:
        return np.nan
    a = a[m]
    b = b[m]
    if a.std() == 0 or b.std() == 0:
        return np.nan
    return float(np.corrcoef(a, b)[0, 1])

def rolling_ar1(x, years, window=30, min_valid=10):
    x = np.asarray(x, float)
    years = np.asarray(years, int)
    n = len(x)
    if n < window:
        window = max(5, n // 2)
    ar1_vals = np.full(n, np.nan, float)
    half = window // 2
    for i in range(n):
        lo = max(0, i - half)
        hi = min(n, i + half + 1)
        seg = x[lo:hi]
        m = np.isfinite(seg)
        seg = seg[m]
        if len(seg) < max(min_valid, 3):
            continue
        a = seg[1:]
        b = seg[:-1]
        if len(a) < 2:
            continue
        ar1_vals[i] = pearson_corr(a, b)
    return years, ar1_vals

def parse_features_string(s):
    feats = []
    for part in str(s).split("|"):
        part = part.strip()
        m = re.search(r"(\w+)\s+EOF(\d+)\s+lag(\d+)", part)
        if not m:
            continue
        var, eofk, lag = m.group(1), int(m.group(2)), int(m.group(3))
        feats.append((var, eofk - 1, lag))  # mode0

    out, seen = [], set()
    for ft in feats:
        if ft not in seen:
            out.append(ft)
            seen.add(ft)
    return out

def parse_spike_ns(path_txt):
    ns = []
    with open(path_txt, "r") as f:
        for line in f:
            m = re.search(r"n=\s*(\d+)", line)
            if m:
                ns.append(int(m.group(1)))
    ns = sorted(set(ns))
    if not ns:
        raise ValueError(f"No spike 'n=' entries found in {path_txt}")
    return ns

def load_ranked_features(csv_path):
    df = pd.read_csv(csv_path)
    for c in ["var", "mode", "lag"]:
        if c not in df.columns:
            raise KeyError(f"Missing {c} in {csv_path}. Columns: {list(df.columns)}")
    df["var"] = df["var"].astype(str).str.strip()
    df["mode0"] = df["mode"].astype(int) - 1
    df["lag"] = df["lag"].astype(int)
    feats = [(r["var"], int(r["mode0"]), int(r["lag"])) for _, r in df.iterrows()]
    return feats, df

def finite_rows_mask(Y, X):
    Y = np.asarray(Y, float)
    X = np.asarray(X, float)
    m = np.isfinite(Y)
    if X.ndim == 1:
        m = m & np.isfinite(X)
    else:
        m = m & np.all(np.isfinite(X), axis=1)
    return m

def safe_r2(y_true, y_pred):
    y_true = np.asarray(y_true, float)
    y_pred = np.asarray(y_pred, float)
    if len(y_true) < 2:
        return np.nan
    return float(r2_score(y_true, y_pred))

def format_lon_label(x, pos=None):
    x = float(x)
    if x < 0:
        return f"{abs(int(x))}°W"
    elif x > 0:
        return f"{int(x)}°E"
    else:
        return "0°"

def format_lat_label(y, pos=None):
    y = float(y)
    if y < 0:
        return f"{abs(int(y))}°S"
    elif y > 0:
        return f"{int(y)}°N"
    else:
        return "0°"

# ============================================================
# LOAD GLOBAL STUFF
# ============================================================
TRAIN_END_YEAR = infer_train_end_year_from_any_eof(EOF_DIR, prefer="so")
print("✅ TRAIN_END_YEAR:", TRAIN_END_YEAR)

amoc_var_used = resolve_amoc_variable(TARGET, AMOC_VARIANT)
print(f"✅ AMOC truth used in figures: {amoc_var_used} (AMOC_VARIANT={AMOC_VARIANT})")

amoc = load_amoc(TARGET, AMOC_FILE, mode=MODE, member_id=member_id, amoc_variant=AMOC_VARIANT)

# Load best subsets per k
df_best = pd.read_csv(BEST_CSV)
df_best = df_best.sort_values("k")
available_ks = sorted(df_best["k"].unique().tolist())
K_MAX_USE = min(K_MAX, int(max(available_ks)) if available_ks else 0)
if K_MAX_USE < 1:
    raise RuntimeError(f"No k rows found in {BEST_CSV}")

print("✅ Will generate k=1..", K_MAX_USE)

# ============================================================
# FUNCTION: build XY for a given feature subset
# ============================================================
def build_XY_for_feats(feats, years, y_amoc, pc_np, global_maxlag=None):
    if global_maxlag is None:
        maxlag = int(max(lag for *_, lag in feats))
    else:
        maxlag = int(global_maxlag)

    used = np.arange(maxlag, len(years), dtype=int)
    years_used = years[used]
    Y = y_amoc[used].astype(float)

    X = np.empty((len(used), len(feats)), float)
    for j, (v, m0, lag) in enumerate(feats):
        X[:, j] = pc_np[v][used - int(lag), int(m0)]

    idx_tr = np.where(years_used <= TRAIN_END_YEAR)[0]
    idx_te = np.where(years_used > TRAIN_END_YEAR)[0]

    if len(idx_tr) < 5 or len(idx_te) < 5:
        raise RuntimeError(
            f"Too few samples after split: train={len(idx_tr)} test={len(idx_te)} "
            f"(years_used={years_used[0]}–{years_used[-1]}, TRAIN_END_YEAR={TRAIN_END_YEAR})"
        )

    train_mask_used = np.zeros(len(years_used), dtype=bool)
    train_mask_used[idx_tr] = True

    if DETREND_Y:
        Y_dt, (ay, by) = detrend_with_train_fit(years_used, Y, train_mask_used)
    else:
        Y_dt = Y.copy()
        ay, by = 0.0, 0.0

    if DETREND_X:
        X_dt = X.copy()
        for j in range(X_dt.shape[1]):
            X_dt[:, j], _ = detrend_with_train_fit(years_used, X[:, j], train_mask_used)
    else:
        X_dt = X.copy()

    return years_used, X_dt, Y_dt, idx_tr, idx_te, (ay, by), used

# ============================================================
# PRELOAD PCs needed for all k (union of features)
# ============================================================
all_feats_union = []
for k in range(1, K_MAX_USE + 1):
    row = df_best.loc[df_best["k"] == k]
    if len(row) != 1:
        continue
    feats = parse_features_string(row.iloc[0]["subset_features"])
    all_feats_union.extend(feats)

needed_vars = sorted(set(v for (v, _, _) in all_feats_union))
pc_dict = {v: load_pc_surface(v, N_MODES, EOF_DIR, mode=MODE, member_id=member_id)
           for v in needed_vars}

common_years = amoc["year"].values.astype(int)
for da in pc_dict.values():
    common_years = np.intersect1d(common_years, da["year"].values.astype(int))
years = np.asarray(common_years, int)
years.sort()
y_amoc = amoc.sel(year=years).values.astype(float)
pc_np = {k: v.sel(year=years).values.astype(float) for k, v in pc_dict.items()}

print(f"✅ Common years: {years[0]}–{years[-1]} (T={len(years)})")

GLOBAL_MAXLAG_FIXED = max(lag for *_, lag in all_feats_union)
print("✅ GLOBAL_MAXLAG_FIXED:", GLOBAL_MAXLAG_FIXED)

# ============================================================
# 1) RECONSTRUCTION FIGURES for k=1..K_MAX_USE
# ============================================================
k_rows = []
pred_store = {}

for k in range(1, K_MAX_USE + 1):
    row = df_best.loc[df_best["k"] == k]
    if len(row) != 1:
        print(f"⚠️ skipping k={k} (not unique)")
        continue

    feats_k = parse_features_string(row.iloc[0]["subset_features"])
    if len(feats_k) == 0:
        print(f"⚠️ skipping k={k} (no feats parsed)")
        continue

    years_used, X_dt, Y_dt, idx_tr, idx_te, (ay, by), used = build_XY_for_feats(
        feats_k, years, y_amoc, pc_np, global_maxlag=GLOBAL_MAXLAG_FIXED
    )

    best_alpha = float(ALPHA_FIXED)
    mdl = make_pipeline(StandardScaler(), Ridge(alpha=float(best_alpha)))

    Xtr, Ytr = X_dt[idx_tr], Y_dt[idx_tr]
    Xte, Yte = X_dt[idx_te], Y_dt[idx_te]

    m_tr = finite_rows_mask(Ytr, Xtr)
    m_te = finite_rows_mask(Yte, Xte)

    if m_tr.sum() < 3 or m_te.sum() < 2:
        print(f"⚠️ k={k}: skipping scoring (finite rows train={m_tr.sum()} test={m_te.sum()})")
        r2_tr = np.nan
        r2_te = np.nan
        c_tr = np.nan
        c_te = np.nan
        pred_tr = np.full(len(idx_tr), np.nan, float)
        pred_te = np.full(len(idx_te), np.nan, float)
    else:
        mdl.fit(Xtr[m_tr], Ytr[m_tr])
        pred_tr = mdl.predict(Xtr)
        pred_te = mdl.predict(Xte)

        r2_tr = safe_r2(Ytr[m_tr], pred_tr[m_tr])
        r2_te = safe_r2(Yte[m_te], pred_te[m_te])
        c_tr  = pearson_corr(Ytr[m_tr], pred_tr[m_tr])
        c_te  = pearson_corr(Yte[m_te], pred_te[m_te])

    k_rows.append({
        "k": k,
        "alpha": best_alpha,
        "train_r2": r2_tr,
        "test_r2": r2_te,
        "train_corr": c_tr,
        "test_corr": c_te,
        "features": " | ".join([f"{v} EOF{m0+1} lag{lag}" for (v, m0, lag) in feats_k]),
        "maxlag": int(max(l for *_, l in feats_k)),
        "global_maxlag": int(GLOBAL_MAXLAG_FIXED),
        "years_used_start": int(years_used[0]),
        "years_used_end": int(years_used[-1]),
    })

    y_pred_all = np.full(len(years), np.nan, float)
    y_pred_all[used[idx_tr]] = pred_tr
    y_pred_all[used[idx_te]] = pred_te

    if DETREND_Y:
        y_true_plot = y_amoc - (ay * years.astype(float) + by)
        y_pred_plot = y_pred_all
        ylab = "AMOC (detrended; TRAIN-fit)"
    else:
        y_true_plot = y_amoc
        y_pred_plot = y_pred_all
        ylab = "AMOC"

    pred_store[k] = dict(
        years=years.copy(),
        y_true=y_true_plot.copy(),
        y_pred=y_pred_plot.copy(),
        ylab=ylab
    )

    info = (
        f"alpha={best_alpha:g}\n"
        f"Train R²={r2_tr:.3f}  r={c_tr:.3f}\n"
        f"Test  R²={r2_te:.3f}  r={c_te:.3f}\n\n"
        "Features:\n" + "\n".join([f"{v} EOF{m0+1} lag{lag}" for (v, m0, lag) in feats_k])
    )

    fig, ax = plt.subplots(figsize=(12, 5))
    ax.plot(years, y_true_plot, label="True AMOC")
    ax.plot(years, y_pred_plot, label="Predicted AMOC")
    ax.axvline(TRAIN_END_YEAR, linestyle="--", label="Split (EOF train end)")
    ax.set_title(f"{MODEL} {TARGET} — reconstruction (k={k})")
    ax.set_xlabel("Year")
    ax.set_ylabel(ylab)
    format_year_axis(ax, years, step=10)
    ax.grid(True, alpha=0.3)

    ax.text(
        0.02, 0.02, info,
        transform=ax.transAxes,
        ha="left", va="bottom",
        fontsize=9,
        bbox=dict(boxstyle="round,pad=0.4", facecolor="white", alpha=0.9, edgecolor="0.2")
    )

    ax.legend(loc="lower left", bbox_to_anchor=(0.28, 0.02), frameon=True)
    fig.tight_layout()

    outbase = os.path.join(OUTDIR, f"01_reconstruction_k{k}")
    savefig(outbase)
    plt.close()
    print("✅ Saved:", outbase + ".png/.pdf")

    print(f"✅ k={k} | alpha={best_alpha:g} | test_R2={r2_te:.4f} | years_used={years_used[0]}–{years_used[-1]}")

df_k = pd.DataFrame(k_rows).sort_values("k").reset_index(drop=True)
df_k.to_csv(os.path.join(OUTDIR, "reconstruction_summary_by_k.csv"), index=False)
print("✅ Saved: reconstruction_summary_by_k.csv")

# ============================================================
# 1b) COMBINED RECONSTRUCTION FIGURE
# ============================================================
if len(pred_store) >= 1:
    ks_plot = sorted(pred_store.keys())

    years0 = pred_store[ks_plot[0]]["years"]
    y_true0 = pred_store[ks_plot[0]]["y_true"]
    ylab0 = pred_store[ks_plot[0]]["ylab"]

    fig, ax = plt.subplots(figsize=(13, 5.2))
    ax.plot(years0, y_true0, linewidth=1.5, label="True AMOC")

    for k in ks_plot:
        ax.plot(
            pred_store[k]["years"],
            pred_store[k]["y_pred"],
            linewidth=1.0,
            label=f"Pred (k={k})"
        )

    ax.axvline(TRAIN_END_YEAR, linestyle="--", label="Split (EOF train end)")
    ax.set_title(f"{MODEL} {TARGET} — reconstructions for k={ks_plot[0]}..{ks_plot[-1]}")
    ax.set_xlabel("Year")
    ax.set_ylabel(ylab0)
    format_year_axis(ax, years0, step=10)
    ax.grid(True, alpha=0.3)
    ax.legend(ncol=2, frameon=True)
    fig.tight_layout()

    outbase = os.path.join(OUTDIR, f"01_reconstruction_ALL_k1to{max(ks_plot)}")
    savefig(outbase)
    plt.close()
    print("✅ Saved:", outbase + ".png/.pdf")

# ============================================================
# 2) MODE/PC plots (deduped)
# ============================================================
mode_sig_to_ks = {}
mode_sig_to_modes = {}

for k in range(1, K_MAX_USE + 1):
    row = df_best.loc[df_best["k"] == k]
    if len(row) != 1:
        print(f"⚠️ skipping MODE/PC k={k} (not unique)")
        continue

    feats_k = parse_features_string(row.iloc[0]["subset_features"])
    if len(feats_k) == 0:
        print(f"⚠️ skipping MODE/PC k={k} (no feats parsed)")
        continue

    uniq_modes_k = []
    seen = set()
    for (v, m0, lag) in feats_k:
        key = (v, int(m0))
        if key not in seen:
            uniq_modes_k.append(key)
            seen.add(key)
    uniq_modes_k = uniq_modes_k[:MAX_MODE_PANELS]

    if len(uniq_modes_k) == 0:
        print(f"⚠️ skipping MODE/PC k={k} (no unique modes)")
        continue

    signature = tuple(uniq_modes_k)
    mode_sig_to_ks.setdefault(signature, []).append(k)
    mode_sig_to_modes.setdefault(signature, uniq_modes_k)

for signature, ks in mode_sig_to_ks.items():
    uniq_modes = mode_sig_to_modes[signature]
    n_pan = len(uniq_modes)

    fig, axes = plt.subplots(
        nrows=n_pan,
        ncols=2,
        figsize=(12, 4.6 * n_pan),
        gridspec_kw={"width_ratios": [0.75, 1.45]}
    )
    if n_pan == 1:
        axes = np.array([axes])

    for i, (v, m0) in enumerate(uniq_modes):
        mode1 = int(m0) + 1
        ax_map = axes[i, 0]
        ax_pc  = axes[i, 1]

        # -------- EOF surface map ----------
        Z, lat2d, lon2d = load_eof_map_surface(v, EOF_DIR, mode_index_1based=mode1)
        vmax = np.nanmax(np.abs(Z))
        if not np.isfinite(vmax) or vmax == 0:
            vmax = 1.0
        norm = mcolors.TwoSlopeNorm(vmin=-vmax, vcenter=0.0, vmax=vmax)

        pcm = ax_map.pcolormesh(lon2d, lat2d, Z, shading="auto", cmap="RdBu_r", norm=norm)

        # Zoom to Atlantic and prevent stretching
        ax_map.set_xlim(-100, 20)
        ax_map.set_ylim(-80, 85)
        ax_map.set_aspect("equal", adjustable="box")

        ax_map.set_xlabel("Longitude")
        ax_map.set_ylabel("Latitude")
        ax_map.set_title(f"{v} | EOF mode {mode1}")

        cb = fig.colorbar(pcm, ax=ax_map, fraction=0.046, pad=0.04)
        cb.set_label("EOF amplitude")

        ax_map.xaxis.set_major_locator(mticker.MultipleLocator(20))
        ax_map.yaxis.set_major_locator(mticker.MultipleLocator(20))
        ax_map.xaxis.set_major_formatter(mticker.FuncFormatter(format_lon_label))
        ax_map.yaxis.set_major_formatter(mticker.FuncFormatter(format_lat_label))

        # -------- PC spaghetti/mean ----------
        PCfull = load_pc_surface_with_members(v, N_MODES, EOF_DIR).sel(year=years)
        train_mask_full = years <= TRAIN_END_YEAR

        if "member" in PCfull.dims:
            pcs = PCfull.isel(mode=int(m0))
            pcs = pcs.transpose("year", "member")
            pcs_np = pcs.values.astype(float)

            if DETREND_X:
                pcs_dt = []
                for j in range(pcs_np.shape[1]):
                    s_dt, _ = detrend_with_train_fit(years, pcs_np[:, j], train_mask_full)
                    pcs_dt.append(s_dt)
                pcs_dt = np.vstack(pcs_dt).T
                pcs_mean = np.nanmean(pcs_dt, axis=1)

                ax_pc.plot(years, pcs_dt, linewidth=PC_SPAGHETTI_LW, alpha=PC_SPAGHETTI_ALPHA, color="grey")
                ax_pc.plot(years, pcs_mean, linewidth=PC_MEAN_LW, color="black", label="PC mean")
                ax_pc.set_ylabel("PC (members detrended; TRAIN-fit)")
            else:
                pcs_mean = np.nanmean(pcs_np, axis=1)
                ax_pc.plot(years, pcs_np, linewidth=PC_SPAGHETTI_LW, alpha=PC_SPAGHETTI_ALPHA, color="grey")
                ax_pc.plot(years, pcs_mean, linewidth=PC_MEAN_LW, color="black", label="PC mean")
                ax_pc.set_ylabel("PC (members)")
        else:
            pc1d = PCfull.isel(mode=int(m0)).values.astype(float)
            if DETREND_X:
                pc1d, _ = detrend_with_train_fit(years, pc1d, train_mask_full)
                ax_pc.set_ylabel("PC (detrended; TRAIN-fit)")
            else:
                ax_pc.set_ylabel("PC")
            ax_pc.plot(years, pc1d, linewidth=PC_MEAN_LW, color="black", label="PC")

        ax_pc.axvline(TRAIN_END_YEAR, linestyle="--", alpha=0.8)
        ax_pc.set_xlabel("Year")
        ax_pc.set_title(f"{v} | PC mode {mode1}")
        ax_pc.grid(True, alpha=0.3)
        format_year_axis(ax_pc, years, step=20)
        ax_pc.legend(loc="upper left", frameon=True)

    fig.suptitle(
        f"{MODEL} {TARGET} — surface modes used (k={ks}, unique panels={n_pan})",
        y=1.02
    )
    plt.tight_layout()

    ktag = "k" + "-".join(str(x) for x in ks)
    outbase = os.path.join(OUTDIR, f"02_EOFmap_and_PC_{ktag}_panels{n_pan}")
    savefig(outbase)
    plt.close()
    print("✅ Saved:", outbase + ".png/.pdf")

# ============================================================
# 2b) AR1 plots (grouped)
# ============================================================
mode_signature_to_ks = {}
mode_signature_to_feats = {}

for k in range(1, K_MAX_USE + 1):
    row = df_best.loc[df_best["k"] == k]
    if len(row) != 1:
        print(f"⚠️ skipping AR1 k={k} (not unique)")
        continue

    feats_k = parse_features_string(row.iloc[0]["subset_features"])
    if len(feats_k) == 0:
        print(f"⚠️ skipping AR1 k={k} (no feats parsed)")
        continue

    uniq_modes_k = []
    seen = set()
    for (v, m0, lag) in feats_k:
        key = (v, int(m0))
        if key not in seen:
            uniq_modes_k.append(key)
            seen.add(key)
    uniq_modes_k = uniq_modes_k[:MAX_MODE_PANELS]

    if len(uniq_modes_k) == 0:
        print(f"⚠️ skipping AR1 k={k} (no unique modes)")
        continue

    maxlag_k = int(max(lag for *_, lag in feats_k))
    signature = (tuple(uniq_modes_k), maxlag_k)

    mode_signature_to_ks.setdefault(signature, []).append(k)
    mode_signature_to_feats.setdefault(signature, feats_k)

for (modes_tuple, maxlag_k), ks in mode_signature_to_ks.items():
    feats_rep = mode_signature_to_feats[(modes_tuple, maxlag_k)]
    uniq_modes = list(modes_tuple)
    n_pan = len(uniq_modes)

    years_used, X_dt, Y_dt, idx_tr, idx_te, (ay, by), used = build_XY_for_feats(
        feats_rep, years, y_amoc, pc_np, global_maxlag=GLOBAL_MAXLAG_FIXED
    )
    train_mask_used = years_used <= TRAIN_END_YEAR

    amoc_ar1_years, amoc_ar1_vals = rolling_ar1(
        Y_dt, years_used, window=AR1_WINDOW_YEARS, min_valid=AR1_MIN_VALID
    )

    fig, ax = plt.subplots(figsize=(12, 4.8))

    for (v, m0) in uniq_modes:
        pc_series = pc_np[v][used, int(m0)]

        if DETREND_X:
            pc_series, _ = detrend_with_train_fit(years_used, pc_series, train_mask_used)

        pc_ar1_years, pc_ar1_vals = rolling_ar1(
            pc_series, years_used, window=AR1_WINDOW_YEARS, min_valid=AR1_MIN_VALID
        )

        ax.plot(
            pc_ar1_years, pc_ar1_vals,
            linewidth=1.8,
            label=f"{v} EOF{int(m0)+1} AR1"
        )

    ax.plot(
        amoc_ar1_years, amoc_ar1_vals,
        linewidth=2.0,
        linestyle="--",
        label="AMOC AR1"
    )

    ax.axvline(TRAIN_END_YEAR, linestyle=":", alpha=0.9)
    ax.set_ylim(-0.5, 1.0)
    ax.grid(True, alpha=0.3)
    ax.set_xlabel("Year")
    ax.set_ylabel("Lag-1 autocorrelation (AR1)")

    ax.set_title(
        f"{MODEL} {TARGET} — Rolling AR(1) (window={AR1_WINDOW_YEARS}y)\n"
        f"k={ks} | unique panels={n_pan} | maxlag={maxlag_k}"
    )

    format_year_axis(ax, years_used, step=10)
    ax.legend(loc="upper left", frameon=True)
    fig.tight_layout()

    ktag = "k" + "-".join(str(x) for x in ks)
    outbase = os.path.join(OUTDIR, f"03_AR1_{ktag}_panels{n_pan}_maxlag{maxlag_k}_W{AR1_WINDOW_YEARS}")
    savefig(outbase)
    plt.close()
    print("✅ Saved:", outbase + ".png/.pdf")

# ============================================================
# 3) HEATMAP for ALL spike modes mentioned in SPIKES_TXT
# ============================================================
feats_ranked, df_rank = load_ranked_features(RANK_CSV)
spike_ns = parse_spike_ns(SPIKES_TXT)

cand_feats = []
for n in spike_ns:
    if 1 <= n <= len(feats_ranked):
        cand_feats.append(feats_ranked[n - 1])

modes_uniq = []
seen = set()
for (v, m0, lag) in cand_feats:
    key = (v, int(m0))
    if key not in seen:
        modes_uniq.append(key)
        seen.add(key)

if len(modes_uniq) == 0:
    raise RuntimeError("No spike modes parsed from SPIKES_TXT + ranking file.")

needed_vars_hm = sorted(set(v for (v, m0) in modes_uniq))
pc_dict_hm = {v: load_pc_surface(v, N_MODES, EOF_DIR, mode=MODE, member_id=member_id)
              for v in needed_vars_hm}

common_years_hm = amoc["year"].values.astype(int)
for da in pc_dict_hm.values():
    common_years_hm = np.intersect1d(common_years_hm, da["year"].values.astype(int))
years_hm = np.asarray(common_years_hm, int)
years_hm.sort()
y_hm = amoc.sel(year=years_hm).values.astype(float)

train_mask_full_hm = years_hm <= TRAIN_END_YEAR

if DETREND_Y:
    y_dt, _ = detrend_with_train_fit(years_hm, y_hm, train_mask_full_hm)
else:
    y_dt = y_hm.copy()

if CORR_ON == "train":
    corr_idx_full = np.where(train_mask_full_hm)[0]
elif CORR_ON == "all":
    corr_idx_full = np.arange(len(years_hm), dtype=int)
else:
    raise ValueError("CORR_ON must be 'train' or 'all'")

lags = np.arange(0, LAG_MAX + 1, dtype=int)
Mhm = len(modes_uniq)
C = np.full((Mhm, len(lags)), np.nan, float)

for i, (v, m0) in enumerate(modes_uniq):
    PC = pc_dict_hm[v].sel(year=years_hm)
    pc = PC.isel(mode=int(m0)).values.astype(float)

    if DETREND_X:
        pc_dt, _ = detrend_with_train_fit(years_hm, pc, train_mask_full_hm)
    else:
        pc_dt = pc.copy()

    for j, lag in enumerate(lags):
        t = corr_idx_full
        t = t[t - lag >= 0]
        if len(t) < 5:
            continue
        a = pc_dt[t - lag]
        b = y_dt[t]
        C[i, j] = pearson_corr(a, b)

mode_labels = [f"{v} EOF{m0+1}" for (v, m0) in modes_uniq]
df_hm = pd.DataFrame(C, index=mode_labels, columns=[f"lag{L}" for L in lags])
csv_out = os.path.join(OUTDIR, f"04_spike_modes_lagcorr_table_top10_L{LAG_MAX}_{CORR_ON}.csv")
df_hm.to_csv(csv_out)
print("✅ Saved:", csv_out)

C_abs = np.abs(C)
vmax = np.nanmax(C_abs)
if not np.isfinite(vmax) or vmax == 0:
    vmax = 1.0

fig, ax = plt.subplots(figsize=(1.0 * len(lags) + 4.5, 0.55 * Mhm + 2.5))
im = ax.imshow(C_abs, aspect="auto", vmin=0, vmax=vmax, cmap=plt.cm.Reds)

ax.set_yticks(np.arange(Mhm))
ax.set_yticklabels(mode_labels)
ax.set_xticks(np.arange(len(lags)))
ax.set_xticklabels([str(L) for L in lags])
ax.set_xlabel("Lag (years)")
ax.set_ylabel("Spike modes (unique)")
ax.set_title(f"{MODEL} {TARGET} | |corr( PC(t−lag), AMOC(t) )| for TOP10 spike modes ({CORR_ON})")

cbar = fig.colorbar(im, ax=ax, shrink=0.9)
cbar.set_label("|corr|")

fig.tight_layout()
outbase = os.path.join(OUTDIR, f"04_spike_modes_lagcorr_heatmap_top10_L{LAG_MAX}_{CORR_ON}")
savefig(outbase)
plt.close()
print("✅ Saved:", outbase + ".png/.pdf")

# ============================================================
# 4) CONTRIBUTION PLOT
# ============================================================
df_k = df_k.sort_values("k").reset_index(drop=True)
Kc = len(df_k)

if Kc >= 1:
    r2s = df_k["test_r2"].values.astype(float)

    deltas = np.empty_like(r2s)
    deltas[0] = r2s[0]
    for i in range(1, len(r2s)):
        deltas[i] = r2s[i] - r2s[i - 1]

    df_contrib = df_k[["k", "alpha", "test_r2"]].copy()
    df_contrib["delta_test_r2"] = deltas
    contrib_csv = os.path.join(OUTDIR, "05_r2_contribution_by_k.csv")
    df_contrib.to_csv(contrib_csv, index=False)
    print("✅ Saved:", contrib_csv)

    step_colors = [
        "tab:blue", "tab:orange", "tab:green", "tab:red", "tab:purple",
        "tab:brown", "tab:pink", "tab:olive", "tab:cyan"
    ]

    x = np.arange(1, Kc + 1)
    fig, ax = plt.subplots(figsize=(10.5, 4.8))

    for i in range(Kc):
        if i > 0 and deltas[i] < 0:
            ax.bar(x[i], r2s[i], color="0.6", edgecolor="0.1")
            ax.text(
                x[i], r2s[i],
                f"{r2s[i]:.3f}",
                ha="center",
                va="bottom" if r2s[i] >= 0 else "top",
                fontsize=7
            )
            continue

        bottom = 0.0
        for j in range(i + 1):
            h = deltas[j]
            if h <= 0:
                continue
            col = step_colors[j % len(step_colors)]
            ax.bar(x[i], h, bottom=bottom, color=col, edgecolor="0.1")
            bottom += h

        ax.text(
            x[i], r2s[i],
            f"{r2s[i]:.3f}",
            ha="center",
            va="bottom" if r2s[i] >= 0 else "top",
            fontsize=7
        )

    ax.axhline(0, linewidth=1)
    ax.set_xticks(x)
    ax.set_xticklabels([f"k={int(k)}" for k in df_k["k"].values])
    ax.set_xlabel("Model size (k features)")
    ax.set_ylabel("TEST R² (cumulative)")
    ax.set_title(f"{MODEL} {TARGET} | Stacked contributions to TEST R² (k=1..{Kc})")
    ax.grid(True, axis="y", alpha=0.25)

    handles = []
    labels = []
    for j in range(min(Kc, len(step_colors))):
        handles.append(plt.Rectangle((0, 0), 1, 1, color=step_colors[j], ec="0.2"))
        labels.append(f"Contribution step {j+1}")
    handles.append(plt.Rectangle((0, 0), 1, 1, color="0.6", ec="0.2"))
    labels.append("Decrease (grey bar)")
    ax.legend(handles, labels, loc="best", frameon=True)

    fig.tight_layout()
    outbase = os.path.join(OUTDIR, "05_r2_contribution_waterfall_by_k_stacked")
    savefig(outbase)
    plt.close()
    print("✅ Saved:", outbase + ".png/.pdf")

✅ TRAIN_END_YEAR: 1989
✅ AMOC truth used in figures: AMOC_45N_ensmean (AMOC_VARIANT=normal)
✅ Will generate k=1.. 3
✅ Common years: 1850–2014 (T=165)
✅ GLOBAL_MAXLAG_FIXED: 12
✅ Saved: /data/users/frekle/Final_figures/CESM2/AMOC_45N_ensmean/Best_test_R2_surface/Not_detrended/Selected_on_train/amoc_variant_normal/minlag_4/01_reconstruction_k1.png/.pdf
✅ k=1 | alpha=1 | test_R2=-0.3033 | years_used=1862–2014
✅ Saved: /data/users/frekle/Final_figures/CESM2/AMOC_45N_ensmean/Best_test_R2_surface/Not_detrended/Selected_on_train/amoc_variant_normal/minlag_4/01_reconstruction_k2.png/.pdf
✅ k=2 | alpha=1 | test_R2=0.3197 | years_used=1862–2014
✅ Saved: /data/users/frekle/Final_figures/CESM2/AMOC_45N_ensmean/Best_test_R2_surface/Not_detrended/Selected_on_train/amoc_variant_normal/minlag_4/01_reconstruction_k3.png/.pdf
✅ k=3 | alpha=1 | test_R2=0.4236 | years_used=1862–2014
✅ Saved: reconstruction_summary_by_k.csv
✅ Saved: /data/users/frekle/Final_figures/CESM2/AMOC_45N_ensmean/Best_test_R2_surfa

/tmp/ipykernel_3442744/1513476089.py:721: UserWarning: The input coordinates to pcolormesh are interpreted as cell centers, but are not monotonically increasing or decreasing. This may lead to incorrectly calculated cell edges, in which case, please supply explicit cell edges to pcolormesh.
  pcm = ax_map.pcolormesh(lon2d, lat2d, Z, shading="auto", cmap="RdBu_r", norm=norm)


✅ Saved: /data/users/frekle/Final_figures/CESM2/AMOC_45N_ensmean/Best_test_R2_surface/Not_detrended/Selected_on_train/amoc_variant_normal/minlag_4/02_EOFmap_and_PC_k1-2_panels1.png/.pdf


/tmp/ipykernel_3442744/1513476089.py:721: UserWarning: The input coordinates to pcolormesh are interpreted as cell centers, but are not monotonically increasing or decreasing. This may lead to incorrectly calculated cell edges, in which case, please supply explicit cell edges to pcolormesh.
  pcm = ax_map.pcolormesh(lon2d, lat2d, Z, shading="auto", cmap="RdBu_r", norm=norm)
/tmp/ipykernel_3442744/1513476089.py:721: UserWarning: The input coordinates to pcolormesh are interpreted as cell centers, but are not monotonically increasing or decreasing. This may lead to incorrectly calculated cell edges, in which case, please supply explicit cell edges to pcolormesh.
  pcm = ax_map.pcolormesh(lon2d, lat2d, Z, shading="auto", cmap="RdBu_r", norm=norm)


✅ Saved: /data/users/frekle/Final_figures/CESM2/AMOC_45N_ensmean/Best_test_R2_surface/Not_detrended/Selected_on_train/amoc_variant_normal/minlag_4/02_EOFmap_and_PC_k3_panels2.png/.pdf
✅ Saved: /data/users/frekle/Final_figures/CESM2/AMOC_45N_ensmean/Best_test_R2_surface/Not_detrended/Selected_on_train/amoc_variant_normal/minlag_4/03_AR1_k1_panels1_maxlag11_W30.png/.pdf
✅ Saved: /data/users/frekle/Final_figures/CESM2/AMOC_45N_ensmean/Best_test_R2_surface/Not_detrended/Selected_on_train/amoc_variant_normal/minlag_4/03_AR1_k2_panels1_maxlag12_W30.png/.pdf
✅ Saved: /data/users/frekle/Final_figures/CESM2/AMOC_45N_ensmean/Best_test_R2_surface/Not_detrended/Selected_on_train/amoc_variant_normal/minlag_4/03_AR1_k3_panels2_maxlag11_W30.png/.pdf
✅ Saved: /data/users/frekle/Final_figures/CESM2/AMOC_45N_ensmean/Best_test_R2_surface/Not_detrended/Selected_on_train/amoc_variant_normal/minlag_4/04_spike_modes_lagcorr_table_top10_L20_train.csv
✅ Saved: /data/users/frekle/Final_figures/CESM2/AMOC_45N_en